# Functional Component Ablation in Hybrid Language Models

**Paper**: *Functional Component Ablation Reveals Specialization Patterns in Hybrid Language Model Architectures*

**Models**: Qwen3.5-0.8B (sequential hybrid), Falcon-H1-0.5B (parallel hybrid), Qwen2.5-0.5B (Transformer control)

## Overview

This notebook implements the complete experimental pipeline:

| Section | Description | Compute |
| ------- | ----------- | ------- |
| **0** | Setup, config, merge Falcon results from HuggingFace | 2 min |
| **1** | Architecture discovery and model loading | 5 min |
| **2** | Ablation mechanism (sequential skip / parallel zeroing) | 1 min |
| **3** | Experiment 1: group, layer-wise, positional ablations + perplexity | 4-8 hrs |
| **4** | Experiment 2: hidden-state contribution metrics | 20 min |
| **5** | Experiment 3: task-dependent statistical analysis | 1 min |
| **6** | Publication tables and LaTeX exports | 1 min |
| **7** | Final manifest and optional appendix experiments | 5 min |

**Hardware**: Google Colab Pro (L4 GPU, 16 GB VRAM). Falcon requires mamba-ssm CUDA kernels.

**Checkpointing**: Every experiment is checkpointed. Sessions can be interrupted and resumed.


In [ ]:
# --- Cell 0A: Setup / package installation ---
# What this cell does:
# 1. Installs the latest Transformers from main (important for Qwen 3.5 support).
# 2. Installs the core scientific stack and optional lm-eval.
# 3. Leaves the notebook runnable in pure Python (no Colab shell magics required).
#
# Expected output:
# - pip installation logs
# - a final confirmation line

import os
import sys
import subprocess

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

def pip_install(args):
    cmd = [sys.executable, "-m", "pip", "install", "--upgrade"] + list(args)
    print("Installing:", " ".join(cmd))
    subprocess.check_call(cmd)

# Core research stack
core_packages = [
    "git+https://github.com/huggingface/transformers.git@main",
    "accelerate>=0.31.0",
    "datasets>=2.20.0",
    "evaluate>=0.4.2",
    "pandas>=2.2.0",
    "numpy>=1.26.0",
    "scipy>=1.12.0",
    "matplotlib>=3.8.0",
    "seaborn>=0.13.0",
    "tqdm>=4.66.0",
    "sentencepiece>=0.2.0",
    "tabulate>=0.9.0",
    "psutil>=5.9.8",
]

pip_install(core_packages)

# Mamba SSM CUDA kernels — critical for Falcon-H1.
# Without these, transformers falls back to torch_forward which is O(n²) in memory
# and will OOM on sequences > ~1024 tokens even on L4/T4 GPUs.
# With the kernels installed, memory is O(n) and full-length prompts fit easily.
MAMBA_KERNELS_AVAILABLE = False
try:
    pip_install(["causal-conv1d>=1.4.0", "mamba-ssm>=2.2.0"])
    MAMBA_KERNELS_AVAILABLE = True
    print("✅ Mamba CUDA kernels installed — Falcon-H1 will use O(n) memory path.")
except Exception as exc:
    print(f"⚠️ Mamba CUDA kernels not installed ({exc}). "
          f"Falcon-H1 will fall back to O(n²) torch_forward — sequence length will be capped.")

# Optional: lm-evaluation-harness cross-checks.
# The notebook defaults to the manual evaluation path below because active hooks
# are easier to control there. If lm-eval fails to install, the notebook continues.
try:
    pip_install(["lm-eval>=0.4.5"])
    print("lm-eval installed successfully.")
except Exception as exc:
    print(f"lm-eval installation failed, continuing without it: {exc}")

print("✅ Package installation completed.")

*Restart the runtime before continuing.*

In [ ]:
# --- Cell 0B: Mount Google Drive, create directories, and define global config ---
# What this cell does:
# 1. Mounts Google Drive when running inside Colab
# 2. Creates the full project directory structure
# 3. Defines all top-level configuration flags
#
# Expected output:
# - Google Drive mount prompt (in Colab)
# - confirmation of created directories
# - a printed configuration summary
# Fallback if Cell 0A was skipped or mamba not installed
if "MAMBA_KERNELS_AVAILABLE" not in dir():
    MAMBA_KERNELS_AVAILABLE = False
import os
import gc
import json
import math
import time
import pickle
import random
import hashlib
import traceback
import warnings
import re
import importlib
import importlib.metadata as importlib_metadata
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn

from tqdm.auto import tqdm
from datasets import load_dataset, concatenate_datasets, get_dataset_config_names
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig, set_seed
from IPython.display import display

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
else:
    print("Not running inside Google Colab; assuming local execution.")

BASE_DIR = "./outputs" if IN_COLAB else "/tmp/2-LinealComponents"
CHECKPOINTS_DIR = os.path.join(BASE_DIR, "checkpoints")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
FIGURES_DIR = os.path.join(BASE_DIR, "figures")
TABLES_DIR = os.path.join(BASE_DIR, "tables")
LOGS_DIR = os.path.join(BASE_DIR, "logs")
ARTIFACTS_DIR = os.path.join(BASE_DIR, "artifacts")

for d in [BASE_DIR, CHECKPOINTS_DIR, RESULTS_DIR, FIGURES_DIR, TABLES_DIR, LOGS_DIR, ARTIFACTS_DIR]:
    os.makedirs(d, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16

# ============================================================================
# EXECUTION GUIDE
# ============================================================================
#
# This notebook supports three execution modes. Pick one and set the flags below.
#
# MODE 1 — SMOKE TEST (~ 30 min on L4)
#   First run. Verifies that architecture discovery, ablation hooks, and
#   benchmarks work end-to-end before committing real compute.
#     QUICK_EVAL  = True
#     PAPER_MODE  = False
#
# MODE 2 — PAPER-CRITICAL (~ 4-6 h on L4)
#   Runs full-quality evaluation ONLY on the sections that appear in the paper:
#   group ablations, random controls, WikiText-2 perplexity, and layer-sweep
#   heatmaps. Layer sweeps use reduced samples to save time because the
#   heatmaps are qualitative. Progressive/positional ablations stay in quick
#   mode unless you explicitly need them.
#     QUICK_EVAL  = True   (still True — PAPER_MODE overrides the critical paths)
#     PAPER_MODE  = True
#
# MODE 3 — FULL (~ 10-12 h on L4)
#   Everything at maximum quality. Only needed if a reviewer asks for it.
#     QUICK_EVAL  = False
#     PAPER_MODE  = True   (redundant but harmless)
#
# RESUMABILITY: every experiment is checkpointed to Google Drive. If Colab
# disconnects, just re-run from the top — completed experiments are loaded
# from cache automatically. You can verify saved checkpoints at any time:
#     print(ckpt.list_checkpoints())
#
# RECOMMENDED WORKFLOW:
#   1. Run MODE 1 once, inspect outputs, fix any issues.
#   2. Switch to MODE 2, re-run. Only uncached experiments will execute.
#   3. If needed, switch to MODE 3 for the final camera-ready pass.
# ============================================================================

# User-requested headline flags
QUICK_EVAL = True   # Set to False for full paper-quality evaluations on ALL sections
PAPER_MODE = True    # Set to True to run paper-critical sections at full quality (overrides QUICK_EVAL selectively)
MODELS_TO_RUN = ["qwen3.5-0.8b", "falcon-h1-0.5b"]

# Paper-mode overrides: full quality only where it matters for publication
_full = (not QUICK_EVAL) or PAPER_MODE
MAX_EVAL_SAMPLES = None if _full else 100                        # group ablations -> paper Table 1
MAX_LAYER_ABLATION_SAMPLES = (50 if PAPER_MODE and QUICK_EVAL else None) if _full else 24   # layer sweeps -> heatmaps (qualitative, 50 is enough)
MAX_PROGRESSIVE_ABLATION_SAMPLES = None if (not QUICK_EVAL) else 40   # progressive ablations -- full only in MODE 3
MAX_GSM8K_SAMPLES = None if _full else 40
DIAGNOSTIC_NUM_TOKENS = 512 if _full else 256

# Recommended extra controls for compute efficiency
USE_BASE_MODELS = True  # cleaner for architecture analysis; set False to use post-trained variants
RUN_OPTIONAL_CONTEXT_STRESS = _full   # auto-enable in paper/full modes
RUN_LM_EVAL_CROSSCHECK = False  # optional baseline-only cross-check
SAVE_EVERY_N_EXAMPLES = 10

# Default sequence-length cap for inference. Models with known OOM risks
# (e.g. Falcon-H1 Mamba SSM quadratic expansion) override this via
# MODEL_SPECS[key]["max_seq_length_no_kernel"]. This cap is ONLY applied
# when the Mamba CUDA kernels are NOT installed. With the kernels,
# memory is O(n) and no cap is needed.
DEFAULT_MAX_INFERENCE_SEQ_LENGTH = 32768  # effectively no cap

def get_max_inference_seq_length(model_key: str) -> int:
    """Return the per-model sequence cap.

    If Mamba CUDA kernels are installed, the cap is ignored because
    the optimized kernel uses O(n) memory instead of O(n²).
    """
    if MAMBA_KERNELS_AVAILABLE:
        return DEFAULT_MAX_INFERENCE_SEQ_LENGTH  # no cap needed
    return MODEL_SPECS.get(model_key, {}).get(
        "max_seq_length_no_kernel", DEFAULT_MAX_INFERENCE_SEQ_LENGTH
    )

# Benchmarks run in the full group-ablation section
BENCHMARKS_MAIN = ["mmlu", "gsm8k", "arc_challenge", "hellaswag", "truthfulqa_mc"]

# For expensive per-layer sweeps, quick mode uses a reduced but still representative set.
LAYER_SWEEP_BENCHMARKS = BENCHMARKS_MAIN if _full else ["mmlu", "arc_challenge"]

# Progressive Qwen ablation (linear layers only)
PROGRESSIVE_ABLATION_STEPS = [1, 2, 4, 6, 8, 12, 16, 18] if (not QUICK_EVAL) else [1, 2, 4, 6, 8, 12, 18]
PROGRESSIVE_ABLATION_BENCHMARKS = BENCHMARKS_MAIN if (not QUICK_EVAL) else ["mmlu", "hellaswag"]

# Falcon positional ablations
FALCON_POSITION_BUCKETS = 3  # early / middle / late

CONFIG_SNAPSHOT = {
    "BASE_DIR": BASE_DIR,
    "CHECKPOINTS_DIR": CHECKPOINTS_DIR,
    "RESULTS_DIR": RESULTS_DIR,
    "FIGURES_DIR": FIGURES_DIR,
    "TABLES_DIR": TABLES_DIR,
    "LOGS_DIR": LOGS_DIR,
    "ARTIFACTS_DIR": ARTIFACTS_DIR,
    "SEED": SEED,
    "DEVICE": DEVICE,
    "DTYPE": str(DTYPE),
    "QUICK_EVAL": QUICK_EVAL,
    "PAPER_MODE": PAPER_MODE,
    "MODELS_TO_RUN": MODELS_TO_RUN,
    "MAX_EVAL_SAMPLES": MAX_EVAL_SAMPLES,
    "MAX_LAYER_ABLATION_SAMPLES": MAX_LAYER_ABLATION_SAMPLES,
    "MAX_PROGRESSIVE_ABLATION_SAMPLES": MAX_PROGRESSIVE_ABLATION_SAMPLES,
    "MAX_GSM8K_SAMPLES": MAX_GSM8K_SAMPLES,
    "DIAGNOSTIC_NUM_TOKENS": DIAGNOSTIC_NUM_TOKENS,
    "USE_BASE_MODELS": USE_BASE_MODELS,
    "RUN_OPTIONAL_CONTEXT_STRESS": RUN_OPTIONAL_CONTEXT_STRESS,
    "MAMBA_KERNELS_AVAILABLE": MAMBA_KERNELS_AVAILABLE,
}

print(json.dumps(CONFIG_SNAPSHOT, indent=2))

if "MAMBA_KERNELS_AVAILABLE" not in dir():
    MAMBA_KERNELS_AVAILABLE = False
    print("\n✅ Mamba CUDA kernels detected → Falcon-H1 runs with O(n) memory, no sequence cap.")
else:
    print(f"\n⚠️ Mamba CUDA kernels NOT available → Falcon-H1 capped at 1024 tokens (O(n²) fallback).")
    print("   To remove the cap: pip install causal-conv1d>=1.4.0 mamba-ssm>=2.2.0")

In [ ]:
# --- Cell 0C: Logging, checkpointing, and export helpers ---
# What this cell does:
# 1. Defines a robust CheckpointManager that saves to Google Drive
# 2. Adds helper utilities for figures, tables, logs, JSON, and cleanup
# 3. Makes every later section resumable after Colab disconnections
#
# Expected output:
# - creation of the log file path
# - checkpoint manager ready message

from datetime import datetime, timezone
from glob import glob

class CheckpointManager:
    def __init__(self, checkpoint_dir: str, log_dir: str):
        self.checkpoint_dir = checkpoint_dir
        self.log_dir = log_dir
        os.makedirs(self.checkpoint_dir, exist_ok=True)
        os.makedirs(self.log_dir, exist_ok=True)
        self.log_path = os.path.join(self.log_dir, "run_log.txt")

    def _pkl_path(self, experiment_name: str) -> str:
        return os.path.join(self.checkpoint_dir, f"{experiment_name}.pkl")

    def _json_path(self, experiment_name: str) -> str:
        return os.path.join(self.checkpoint_dir, f"{experiment_name}.json")

    def log(self, message: str):
        stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        line = f"[{stamp}] {message}"
        print(line)
        with open(self.log_path, "a", encoding="utf-8") as f:
            f.write(line + "\n")

    def save(self, experiment_name: str, data):
        path = self._pkl_path(experiment_name)
        tmp_path = path + ".tmp"
        with open(tmp_path, "wb") as f:
            pickle.dump(data, f)
        os.replace(tmp_path, path)
        # Force filesystem flush to Google Drive to minimise data loss on disconnect
        try:
            os.fsync(os.open(path, os.O_RDONLY))
        except Exception:
            pass
        self._save_count = getattr(self, "_save_count", 0) + 1

    def load(self, experiment_name: str):
        path = self._pkl_path(experiment_name)
        if not os.path.exists(path):
            return None
        try:
            with open(path, "rb") as f:
                return pickle.load(f)
        except (pickle.UnpicklingError, EOFError, OSError) as exc:
            self.log(f"[WARNING] Corrupt checkpoint {experiment_name}: {exc}. Skipping.")
            return None

    def exists(self, experiment_name: str) -> bool:
        return os.path.exists(self._pkl_path(experiment_name))

    def save_json(self, experiment_name: str, data):
        path = self._json_path(experiment_name)
        tmp_path = path + ".tmp"
        with open(tmp_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        os.replace(tmp_path, path)

    def load_json(self, experiment_name: str):
        path = self._json_path(experiment_name)
        if not os.path.exists(path):
            return None
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)

    def list_checkpoints(self, prefix: str | None = None):
        pattern = os.path.join(self.checkpoint_dir, "*.pkl")
        files = sorted(glob(pattern))
        if prefix is not None:
            files = [f for f in files if os.path.basename(f).startswith(prefix)]
        return files

ckpt = CheckpointManager(CHECKPOINTS_DIR, LOGS_DIR)

# Startup: report existing checkpoints so the user knows what will be reused
_existing_ckpts = ckpt.list_checkpoints()
if _existing_ckpts:
    print(f"Found {len(_existing_ckpts)} existing checkpoint(s) on Drive — completed experiments will be loaded from cache.")
    for _p in _existing_ckpts:
        _name = os.path.basename(_p)
        _size_kb = os.path.getsize(_p) / 1024
        print(f"  {_name} ({_size_kb:.1f} KB)")
else:
    print("No existing checkpoints found — this is a fresh run.")

def now_utc():
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

def count_parameters(module: nn.Module) -> int:
    return sum(p.numel() for p in module.parameters())

def count_trainable_parameters(module: nn.Module) -> int:
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

def safe_float(x):
    if x is None:
        return None
    try:
        return float(x)
    except Exception:
        return x

def save_dataframe(df: pd.DataFrame, name: str, index: bool = False):
    csv_path = os.path.join(RESULTS_DIR, f"{name}.csv")
    tex_path = os.path.join(TABLES_DIR, f"{name}.tex")
    df.to_csv(csv_path, index=index)
    with open(tex_path, "w", encoding="utf-8") as f:
        f.write(df.to_latex(index=index, escape=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))
    return {"csv": csv_path, "tex": tex_path}

def save_text(text: str, path: str):
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)

def save_json_file(obj, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def save_figure(fig: plt.Figure, name: str, dpi: int = 300):
    png_path = os.path.join(FIGURES_DIR, f"{name}.png")
    pdf_path = os.path.join(FIGURES_DIR, f"{name}.pdf")
    fig.tight_layout()
    fig.savefig(png_path, dpi=dpi, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.close(fig)
    return {"png": png_path, "pdf": pdf_path}

def cleanup_model(model=None, tokenizer=None):
    try:
        if model is not None:
            del model
        if tokenizer is not None:
            del tokenizer
    except Exception:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def hash_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]

ckpt.log("Checkpoint manager initialized.")
print(f"Log file: {ckpt.log_path}")

import os
import json
import sys
import platform
import numpy as np
import pandas as pd
import torch

try:
    from importlib import metadata as importlib_metadata
except ImportError:
    import importlib_metadata

from datetime import datetime, timezone

def now_utc():
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

def get_gpu_info():
    info = {
        "cuda_available": torch.cuda.is_available(),
        "device_count": torch.cuda.device_count(),
    }
    if torch.cuda.is_available():
        info["device_name"] = torch.cuda.get_device_name(0)
        props = torch.cuda.get_device_properties(0)
        info["total_memory_gb"] = round(props.total_memory / (1024 ** 3), 2)
        info["capability"] = f"{props.major}.{props.minor}"
        info["bf16_supported"] = torch.cuda.is_bf16_supported()
    return info

def get_package_version(name: str):
    try:
        return importlib_metadata.version(name)
    except Exception:
        return None

ENVIRONMENT = {
    "timestamp": now_utc(),
    "python": sys.version,
    "platform": platform.platform(),
    "pytorch": torch.__version__,
    "transformers": get_package_version("transformers"),
    "datasets": get_package_version("datasets"),
    "accelerate": get_package_version("accelerate"),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "matplotlib": get_package_version("matplotlib"),
    "seaborn": get_package_version("seaborn"),
    "lm_eval": get_package_version("lm-eval"),
    "gpu": get_gpu_info(),
    "config": CONFIG_SNAPSHOT,
}

env_path = os.path.join(RESULTS_DIR, "environment_snapshot.json")
save_json_file(ENVIRONMENT, env_path)
ckpt.log(f"Saved environment snapshot to {env_path}")
print(json.dumps(ENVIRONMENT, indent=2))

# Section 0D — Merge Falcon RunPod results

Downloads Falcon Experiment 1 checkpoints from HuggingFace Hub and merges them into the project checkpoint directory alongside the Qwen results already on Google Drive.

In [ ]:
# --- Cell 0D: Download and merge Falcon results from HuggingFace Hub ---
# What this cell does:
# 1. Downloads falcon_results_final.tar.gz from centroIA/paper2-falcon-results
# 2. Extracts all .pkl checkpoint files into CHECKPOINTS_DIR
# 3. Verifies the merge by counting Qwen and Falcon checkpoints
#
# Expected output:
# - download progress bar
# - merge summary showing Qwen + Falcon checkpoint counts

from huggingface_hub import hf_hub_download
import tarfile

# --- CONFIG: set your HF repo here ---
FALCON_HF_REPO = "centroIA/paper2-falcon-results"
FALCON_HF_FILE = "falcon_results_final.tar.gz"
# If your repo is private, uncomment and set your token:
HF_TOKEN_FOR_DOWNLOAD = None  # set to "hf_YOUR_TOKEN" if repo is private

print("Downloading Falcon results from HuggingFace Hub...")
local_tar = hf_hub_download(
    repo_id=FALCON_HF_REPO,
    filename=FALCON_HF_FILE,
    repo_type="dataset",
    local_dir="/tmp",
    token=HF_TOKEN_FOR_DOWNLOAD,
)
print(f"Downloaded: {local_tar}")

# Extract .pkl files into CHECKPOINTS_DIR
# The tar has files both at root (./file.pkl) and inside ./checkpoints/file.pkl
# We want all .pkl files flat in CHECKPOINTS_DIR
extracted = 0
with tarfile.open(local_tar, "r:gz") as tar:
    for member in tar.getmembers():
        if member.name.endswith(".pkl") and not member.isdir():
            # Extract to flat directory regardless of internal path
            member.name = os.path.basename(member.name)
            tar.extract(member, CHECKPOINTS_DIR)
            extracted += 1

# Verify merge
import glob
all_ckpts = glob.glob(os.path.join(CHECKPOINTS_DIR, "*.pkl"))
qwen_ckpts = [f for f in all_ckpts if "qwen" in os.path.basename(f)]
falcon_ckpts = [f for f in all_ckpts if "falcon" in os.path.basename(f)]
other_ckpts = [f for f in all_ckpts if "qwen" not in os.path.basename(f) and "falcon" not in os.path.basename(f)]

print(f"\n✅ Merge complete:")
print(f"   Falcon extracted:   {extracted} files")
print(f"   Qwen checkpoints:   {len(qwen_ckpts)}")
print(f"   Falcon checkpoints: {len(falcon_ckpts)}")
print(f"   Shared checkpoints: {len(other_ckpts)}")
print(f"   Total:              {len(all_ckpts)}")

# Verify Falcon Exp 1 master checkpoint
falcon_master = os.path.join(CHECKPOINTS_DIR, "exp1_master__falcon-h1-0.5b.pkl")
if os.path.exists(falcon_master):
    import pickle
    with open(falcon_master, "rb") as f:
        data = pickle.load(f)
    conditions = data.get("payload", {}).get("conditions", {})
    print(f"\n✅ Falcon Exp 1: status={data.get('status')}, conditions={len(conditions)}")
else:
    print("\n❌ Falcon Exp 1 master checkpoint NOT found!")


# Section 1 — Model loading and architecture exploration

This section loads **one model at a time**, dumps `named_modules()` to disk, discovers the decoder stack,
and builds a structured summary that later sections reuse for ablations and plots.


In [ ]:
# --- Cell 1A: Model registry and loading helpers ---
# What this cell does:
# 1. Defines model IDs and architecture metadata
# 2. Provides safe loader utilities
# 3. Exposes context-length and output-head helpers used later
#
# Expected output:
# - model registry printed as a DataFrame

LETTERS = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
DATASET_CACHE = {}

MODEL_SPECS = {
    "qwen3.5-0.8b": {
        "display_name": "Qwen3.5-0.8B",
        "model_id": "Qwen/Qwen3.5-0.8B-Base" if USE_BASE_MODELS else "Qwen/Qwen3.5-0.8B",
        "arch_family": "qwen_sequential_hybrid",
        "notes": "Sequential hybrid with interleaved linear_attention and full_attention layers.",
    },
    "falcon-h1-0.5b": {
        "display_name": "Falcon-H1-0.5B",
        "model_id": "tiiuae/Falcon-H1-0.5B-Base" if USE_BASE_MODELS else "tiiuae/Falcon-H1-0.5B-Instruct",
        "arch_family": "falcon_parallel_hybrid",
        "max_seq_length_no_kernel": 1024,  # only applies when mamba-ssm CUDA kernels are NOT installed
        "notes": "Parallel hybrid block containing attention and SSM paths in each block.",
    },
}

def get_model_context_length(model) -> int:
    cfg = model.config
    for attr in ["max_position_embeddings", "n_positions", "seq_length", "max_seq_len", "model_max_length", "sliding_window"]:
        val = getattr(cfg, attr, None)
        if isinstance(val, int) and val > 0:
            return int(val)
    return 4096

def get_lm_head(model):
    head = None
    if hasattr(model, "get_output_embeddings"):
        head = model.get_output_embeddings()
    if head is None and hasattr(model, "lm_head"):
        head = model.lm_head
    if head is None:
        raise ValueError("Could not find the LM head / output embedding projection.")
    return head

def get_final_norm_module(model):
    candidates = [
        "model.norm",
        "model.final_layernorm",
        "model.norm_f",
        "transformer.ln_f",
        "transformer.norm",
    ]
    for path in candidates:
        try:
            obj = model
            for part in path.split("."):
                obj = getattr(obj, part)
            if isinstance(obj, nn.Module):
                return obj
        except Exception:
            continue
    return None

def resolve_module_by_path(root: nn.Module, path: str):
    obj = root
    for part in path.split("."):
        if part.isdigit():
            obj = obj[int(part)]
        else:
            obj = getattr(obj, part)
    return obj

def get_decoder_layers(model):
    # Preferred common paths first
    candidates = [
        ("model.layers", lambda m: getattr(getattr(m, "model", None), "layers", None)),
        ("transformer.h", lambda m: getattr(getattr(m, "transformer", None), "h", None)),
        ("gpt_neox.layers", lambda m: getattr(getattr(m, "gpt_neox", None), "layers", None)),
    ]
    for path, fn in candidates:
        layers = fn(model)
        if isinstance(layers, (nn.ModuleList, list)) and len(layers) > 0:
            return layers, path

    # Fallback: choose the largest ModuleList of repeated blocks
    best_name, best_module = None, None
    for name, module in model.named_modules():
        if isinstance(module, nn.ModuleList) and len(module) >= 4:
            if best_module is None or len(module) > len(best_module):
                best_name, best_module = name, module
    if best_module is None:
        raise ValueError("Could not locate decoder layers automatically.")
    return best_module, best_name

def load_model_and_tokenizer(model_key: str):
    spec = MODEL_SPECS[model_key]
    model_id = spec["model_id"]
    ckpt.log(f"Loading model {model_key} from {model_id}")

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True,
        use_fast=True,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token if tokenizer.eos_token is not None else tokenizer.unk_token
    tokenizer.padding_side = "left"

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=DTYPE,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )
    model.eval()
    model.to(DEVICE)

    metadata = {
        "model_key": model_key,
        "model_id": model_id,
        "display_name": spec["display_name"],
        "arch_family": spec["arch_family"],
        "dtype": str(DTYPE),
        "device": DEVICE,
        "context_length": get_model_context_length(model),
        "num_parameters": count_parameters(model),
    }
    return model, tokenizer, metadata

registry_df = pd.DataFrame.from_dict(MODEL_SPECS, orient="index")
display(registry_df)

In [ ]:
# --- Cell 1B: Architecture discovery helpers ---
# What this cell does:
# 1. Normalizes Qwen layer types
# 2. Dynamically discovers Falcon attention / SSM submodules
# 3. Dumps the full model.named_modules() listing to Drive
#
# Expected output:
# - helper functions defined
# - no heavy execution yet

def dump_named_modules(model: nn.Module, model_key: str, max_console_lines: int = 200):
    lines = []
    for name, module in model.named_modules():
        lines.append(f"{name}\t{module.__class__.__name__}")
    path = os.path.join(ARTIFACTS_DIR, f"{model_key}_named_modules.txt")
    save_text("\n".join(lines), path)
    print("\n".join(lines[:max_console_lines]))
    if len(lines) > max_console_lines:
        print(f"... ({len(lines) - max_console_lines} more lines omitted from console)")
    print(f"Full named_modules dump saved to: {path}")
    return path

def normalize_qwen_layer_type(layer_type: str) -> str:
    mapping = {
        "linear_attention": "linear",
        "full_attention": "attention",
        "linear": "linear",
        "attention": "attention",
    }
    return mapping.get(layer_type, layer_type)

def summarize_qwen_architecture(model, model_key: str):
    layers, layer_path = get_decoder_layers(model)
    raw_types = list(getattr(model.config, "layer_types"))
    norm_types = [normalize_qwen_layer_type(x) for x in raw_types]

    rows = []
    linear_indices = []
    attention_indices = []

    for idx, layer in enumerate(layers):
        layer_type = norm_types[idx]
        token_mixer = getattr(layer, "linear_attn", None) if layer_type == "linear" else getattr(layer, "self_attn", None)
        token_mixer_params = count_parameters(token_mixer) if token_mixer is not None else 0
        mlp = getattr(layer, "mlp", None)
        mlp_params = count_parameters(mlp) if mlp is not None else 0
        rows.append({
            "layer_idx": idx,
            "layer_type_raw": raw_types[idx],
            "layer_type": layer_type,
            "layer_class": layer.__class__.__name__,
            "token_mixer_params": token_mixer_params,
            "mlp_params": mlp_params,
            "total_layer_params": count_parameters(layer),
        })
        if layer_type == "linear":
            linear_indices.append(idx)
        elif layer_type == "attention":
            attention_indices.append(idx)

    df = pd.DataFrame(rows)
    summary = {
        "model_key": model_key,
        "layer_container_path": layer_path,
        "num_layers": len(rows),
        "linear_indices": linear_indices,
        "attention_indices": attention_indices,
        "layer_types_raw": raw_types,
        "layer_types_norm": norm_types,
        "rows": rows,
    }
    return df, summary

def choose_best_component_candidate(candidates):
    if not candidates:
        return None
    # Prefer modules with the largest parameter count; break ties with shallower paths.
    candidates = sorted(candidates, key=lambda x: (-x["num_params"], x["depth"], x["path"]))
    return candidates[0]

def summarize_falcon_architecture(model, model_key: str):
    layers, layer_path = get_decoder_layers(model)
    rows = []
    discovery_rows = []

    for idx, layer in enumerate(layers):
        attn_candidates = []
        ssm_candidates = []

        for name, module in layer.named_modules():
            if name == "":
                continue
            module_name = name.lower()
            class_name = module.__class__.__name__.lower()
            num_params = count_parameters(module)
            if num_params == 0:
                continue

            full_path = f"{layer_path}.{idx}.{name}"
            depth = name.count(".") + 1

            attn_flag = (
                ("attention" in class_name) or
                ("attn" in class_name) or
                ("attention" in module_name) or
                ("attn" in module_name)
            )
            ssm_flag = (
                ("mamba" in class_name) or
                ("ssm" in class_name) or
                ("mamba" in module_name) or
                ("ssm" in module_name)
            )

            if attn_flag:
                attn_candidates.append({
                    "layer_idx": idx,
                    "component_type": "attention",
                    "path": full_path,
                    "depth": depth,
                    "class_name": module.__class__.__name__,
                    "num_params": num_params,
                })

            if ssm_flag:
                ssm_candidates.append({
                    "layer_idx": idx,
                    "component_type": "ssm",
                    "path": full_path,
                    "depth": depth,
                    "class_name": module.__class__.__name__,
                    "num_params": num_params,
                })

        best_attn = choose_best_component_candidate(attn_candidates)
        best_ssm = choose_best_component_candidate(ssm_candidates)

        rows.append({
            "layer_idx": idx,
            "layer_class": layer.__class__.__name__,
            "attention_path": None if best_attn is None else best_attn["path"],
            "attention_class": None if best_attn is None else best_attn["class_name"],
            "attention_params": 0 if best_attn is None else best_attn["num_params"],
            "ssm_path": None if best_ssm is None else best_ssm["path"],
            "ssm_class": None if best_ssm is None else best_ssm["class_name"],
            "ssm_params": 0 if best_ssm is None else best_ssm["num_params"],
            "total_layer_params": count_parameters(layer),
        })

        discovery_rows.extend(attn_candidates)
        discovery_rows.extend(ssm_candidates)

    df = pd.DataFrame(rows)
    discovery_df = pd.DataFrame(discovery_rows)

    summary = {
        "model_key": model_key,
        "layer_container_path": layer_path,
        "num_layers": len(rows),
        "rows": rows,
        "component_discovery_rows": discovery_rows,
        "attention_paths": {int(r["layer_idx"]): r["attention_path"] for r in rows if r["attention_path"] is not None},
        "ssm_paths": {int(r["layer_idx"]): r["ssm_path"] for r in rows if r["ssm_path"] is not None},
    }
    return df, discovery_df, summary

def run_architecture_exploration(model_key: str):
    experiment_name = f"architecture_analysis__{model_key}"
    cached = ckpt.load(experiment_name)
    if cached is not None:
        ckpt.log(f"✅ {experiment_name} already completed. Loading from checkpoint.")
        return cached

    model, tokenizer, meta = load_model_and_tokenizer(model_key)
    try:
        named_modules_path = dump_named_modules(model, model_key)

        if MODEL_SPECS[model_key]["arch_family"] == "qwen_sequential_hybrid":
            layer_df, summary = summarize_qwen_architecture(model, model_key)
            discovery_df = None
        else:
            layer_df, discovery_df, summary = summarize_falcon_architecture(model, model_key)

        layer_csv = os.path.join(RESULTS_DIR, f"{model_key}_architecture_layers.csv")
        layer_df.to_csv(layer_csv, index=False)

        discovery_csv = None
        if discovery_df is not None and len(discovery_df) > 0:
            discovery_csv = os.path.join(RESULTS_DIR, f"{model_key}_architecture_component_discovery.csv")
            discovery_df.to_csv(discovery_csv, index=False)

        result = {
            "meta": meta,
            "summary": summary,
            "layer_table": layer_df,
            "discovery_table": discovery_df,
            "layer_csv": layer_csv,
            "discovery_csv": discovery_csv,
            "named_modules_path": named_modules_path,
        }
        ckpt.save(experiment_name, result)
        ckpt.log(f"💾 Saved checkpoint for {experiment_name}")
        return result
    finally:
        cleanup_model(model, tokenizer)

In [ ]:
# --- Cell 1B.5: Reload architecture checkpoints after runtime restarts ---
# What this cell does:
# 1. Provides a single helper for reloading architecture analysis from Drive
# 2. Makes later sections robust to Colab disconnects and fresh runtimes
#
# Expected output:
# - helper defined

def get_architecture_result(model_key: str):
    cached = ckpt.load(f"architecture_analysis__{model_key}")
    if cached is not None:
        return cached
    return run_architecture_exploration(model_key)

In [ ]:
# --- Cell 1C: Execute architecture exploration and generate summary figures ---
# What this cell does:
# 1. Loads each selected model one at a time
# 2. Runs structured architecture discovery
# 3. Saves architecture tables and simple diagnostic figures
#
# Expected output:
# - architecture summaries for each selected model
# - figures saved to FIGURES_DIR

architecture_results = {}

for model_key in MODELS_TO_RUN:
    architecture_results[model_key] = run_architecture_exploration(model_key)

    layer_df = architecture_results[model_key]["layer_table"]
    meta = architecture_results[model_key]["meta"]

    # Summary table export
    summary_rows = [{
        "model_key": model_key,
        "model_id": meta["model_id"],
        "display_name": meta["display_name"],
        "arch_family": meta["arch_family"],
        "num_parameters": meta["num_parameters"],
        "context_length": meta["context_length"],
        "num_layers": len(layer_df),
    }]
    summary_df = pd.DataFrame(summary_rows)
    save_dataframe(summary_df, f"{model_key}_architecture_summary", index=False)

    # Figure 1: layer distribution / component params
    fig, ax = plt.subplots(figsize=(10, 3.5))
    x = np.arange(len(layer_df))

    if MODEL_SPECS[model_key]["arch_family"] == "qwen_sequential_hybrid":
        layer_colors = ["C0" if t == "linear" else "C1" for t in layer_df["layer_type"]]
        ax.bar(x, np.ones_like(x), color=layer_colors)
        ax.set_yticks([])
        ax.set_xlabel("Layer index")
        ax.set_title(f"{model_key}: layer type distribution")
        legend_handles = [
            plt.Line2D([0], [0], color="C0", lw=6, label="linear / Gated DeltaNet"),
            plt.Line2D([0], [0], color="C1", lw=6, label="softmax attention"),
        ]
        ax.legend(handles=legend_handles, loc="upper right")
    else:
        ax.plot(x, layer_df["attention_params"] / 1e6, marker="o", label="attention params (M)")
        ax.plot(x, layer_df["ssm_params"] / 1e6, marker="s", label="ssm params (M)")
        ax.set_xlabel("Layer index")
        ax.set_ylabel("Parameters (millions)")
        ax.set_title(f"{model_key}: discovered component parameters by block")
        ax.legend()

    save_figure(fig, f"{model_key}_architecture_overview")

print("Architecture exploration complete.")
for model_key, result in architecture_results.items():
    print("=" * 80)
    print(model_key)
    print(result["layer_table"].head())

# Section 2 — Ablation mechanism implementation

This section defines the reversible ablation machinery:

- **Qwen**: skip whole decoder layers (functional identity through the residual path)
- **Falcon-H1**: zero attention or SSM outputs via forward hooks before the rest of the block consumes them

The design goal is **reversibility** and **minimal intrusion**.


In [ ]:
# --- Cell 2A: Reversible ablation manager ---
# What this cell does:
# 1. Creates a single manager class for patches and hooks
# 2. Supports Qwen layer skipping and Falcon component zeroing
# 3. Exposes the exact helper names requested in the project brief
#
# Expected output:
# - helper class defined
# - no model execution yet

import types

def extract_primary_tensor(output):
    if torch.is_tensor(output):
        return output
    if isinstance(output, (tuple, list)) and len(output) > 0 and torch.is_tensor(output[0]):
        return output[0]
    return None

def zero_primary_tensor_in_output(output):
    if torch.is_tensor(output):
        return torch.zeros_like(output)
    if isinstance(output, tuple):
        if len(output) > 0 and torch.is_tensor(output[0]):
            return (torch.zeros_like(output[0]),) + tuple(output[1:])
        return output
    if isinstance(output, list):
        if len(output) > 0 and torch.is_tensor(output[0]):
            return [torch.zeros_like(output[0])] + list(output[1:])
        return output
    return output

class AblationManager:
    def __init__(self, model, model_key: str, architecture_summary: dict):
        self.model = model
        self.model_key = model_key
        self.architecture_summary = architecture_summary
        self.patched_forwards = {}
        self.hook_handles = []

    def clear(self):
        for key, item in list(self.patched_forwards.items()):
            module = item["module"]
            module.forward = item["original_forward"]
        self.patched_forwards.clear()

        for handle in self.hook_handles:
            try:
                handle.remove()
            except Exception:
                pass
        self.hook_handles = []

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, tb):
        self.clear()

    # -----------------
    # Output-length probing
    # -----------------
    def _probe_layer_output_length(self, layer_idx: int) -> int:
        if not hasattr(self, "_output_lengths"):
            self._output_lengths = {}
        if layer_idx in self._output_lengths:
            return self._output_lengths[layer_idx]

        try:
            layers, _ = get_decoder_layers(self.model)
            layer = layers[layer_idx]
            hidden_size = self.model.config.hidden_size
            device = next(layer.parameters()).device
            dtype = next(layer.parameters()).dtype
            dummy = torch.zeros(1, 1, hidden_size, device=device, dtype=dtype)
            with torch.no_grad():
                out = layer(dummy, use_cache=False)
            if isinstance(out, (tuple, list)):
                length = len(out)
            else:
                length = 1
        except Exception:
            length = 1

        self._output_lengths[layer_idx] = length
        return length

    # -----------------
    # Qwen: skip layers
    # -----------------
    def skip_layer(self, layer_idx: int):
        layers, _ = get_decoder_layers(self.model)
        layer = layers[layer_idx]

        if layer_idx in self.patched_forwards:
            return

        original_forward = layer.forward
        output_len = self._probe_layer_output_length(layer_idx)

        def identity_forward(this_layer, hidden_states, *args, **kwargs):
            if output_len > 1:
                return (hidden_states,) + (None,) * (output_len - 1)
            return hidden_states

        layer.forward = types.MethodType(identity_forward, layer)
        self.patched_forwards[layer_idx] = {
            "module": layer,
            "original_forward": original_forward,
        }

    def restore_layer(self, layer_idx: int):
        if layer_idx not in self.patched_forwards:
            return
        item = self.patched_forwards.pop(layer_idx)
        item["module"].forward = item["original_forward"]

    def skip_all_layers_of_type(self, layer_type: str):
        layer_type = normalize_qwen_layer_type(layer_type)
        layer_types = self.architecture_summary["layer_types_norm"]
        for idx, t in enumerate(layer_types):
            if t == layer_type:
                self.skip_layer(idx)

    # ------------------------------------------
    # Falcon / generic: zero component outputs
    # ------------------------------------------
    def zero_component_output(self, component_type: str, layer_indices=None):
        if layer_indices is None:
            layer_indices = []

        arch_family = MODEL_SPECS[self.model_key]["arch_family"]

        if arch_family == "falcon_parallel_hybrid":
            if component_type == "ssm":
                path_map = self.architecture_summary["ssm_paths"]
            elif component_type == "attention":
                path_map = self.architecture_summary["attention_paths"]
            else:
                raise ValueError(f"Unsupported Falcon component_type: {component_type}")

            target_indices = list(path_map.keys()) if len(layer_indices) == 0 else layer_indices
            for idx in target_indices:
                if idx not in path_map:
                    continue
                module = resolve_module_by_path(self.model, path_map[idx])
                handle = module.register_forward_hook(lambda module, inputs, output: zero_primary_tensor_in_output(output))
                self.hook_handles.append(handle)
        else:
            # Optional generic path: for Qwen this can zero the token mixer submodule output directly if needed.
            layers, _ = get_decoder_layers(self.model)
            target_indices = range(len(layers)) if len(layer_indices) == 0 else layer_indices
            for idx in target_indices:
                layer = layers[idx]
                layer_type = self.architecture_summary["layer_types_norm"][idx]
                if component_type == "linear" and layer_type == "linear" and hasattr(layer, "linear_attn"):
                    handle = layer.linear_attn.register_forward_hook(lambda module, inputs, output: zero_primary_tensor_in_output(output))
                    self.hook_handles.append(handle)
                elif component_type == "attention" and layer_type == "attention" and hasattr(layer, "self_attn"):
                    handle = layer.self_attn.register_forward_hook(lambda module, inputs, output: zero_primary_tensor_in_output(output))
                    self.hook_handles.append(handle)

# Helper names requested by the user brief
def skip_layer(model, layer_idx, model_key, architecture_summary, ablator: AblationManager | None = None):
    if ablator is None:
        raise ValueError("Pass an active AblationManager instance via `ablator=`.")
    ablator.skip_layer(layer_idx)

def restore_layer(model, layer_idx, model_key, architecture_summary, ablator: AblationManager | None = None):
    if ablator is None:
        raise ValueError("Pass an active AblationManager instance via `ablator=`.")
    ablator.restore_layer(layer_idx)

def skip_all_layers_of_type(model, layer_type, model_key, architecture_summary, ablator: AblationManager | None = None):
    if ablator is None:
        raise ValueError("Pass an active AblationManager instance via `ablator=`.")
    ablator.skip_all_layers_of_type(layer_type)

def zero_component_output(model, component_type, model_key, architecture_summary, layer_indices=None, ablator: AblationManager | None = None):
    if ablator is None:
        raise ValueError("Pass an active AblationManager instance via `ablator=`.")
    ablator.zero_component_output(component_type, layer_indices=layer_indices)

In [ ]:
# --- Cell 2B: Ablation verification on a smoke-test prompt ---
# What this cell does:
# 1. Verifies that the ablation mechanism actually changes model outputs
# 2. Saves a small verification checkpoint per model
#
# Expected output:
# - baseline vs ablated logit deltas > 0
# - verification checkpoints saved

def encode_prompt(tokenizer, prompt: str):
    batch = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    return {k: v.to(DEVICE) for k, v in batch.items()}

def top_tokens_from_logits(logits, tokenizer, k: int = 5):
    values, indices = torch.topk(logits, k=k, dim=-1)
    return [(tokenizer.decode([idx.item()]), float(val.item())) for idx, val in zip(indices, values)]

SMOKE_PROMPT = "The capital of France is"

for model_key in MODELS_TO_RUN:
    experiment_name = f"ablation_verification__{model_key}"
    cached = ckpt.load(experiment_name)
    if cached is not None:
        ckpt.log(f"✅ {experiment_name} already completed. Loading from checkpoint.")
        print(model_key, cached)
        continue

    arch = get_architecture_result(model_key)["summary"]
    model, tokenizer, meta = load_model_and_tokenizer(model_key)

    try:
        inputs = encode_prompt(tokenizer, SMOKE_PROMPT)
        with torch.inference_mode():
            baseline = model(**inputs, use_cache=False).logits[0, -1].float().cpu()

        if MODEL_SPECS[model_key]["arch_family"] == "qwen_sequential_hybrid":
            target_idx = arch["linear_indices"][0] if len(arch["linear_indices"]) > 0 else 0
            with AblationManager(model, model_key, arch) as ablator:
                ablator.skip_layer(target_idx)
                with torch.inference_mode():
                    ablated = model(**inputs, use_cache=False).logits[0, -1].float().cpu()
                condition = f"skip_layer_{target_idx}"
        else:
            # zero the first SSM path by default for verification
            target_idx = min(arch["ssm_paths"].keys()) if len(arch["ssm_paths"]) > 0 else 0
            with AblationManager(model, model_key, arch) as ablator:
                ablator.zero_component_output("ssm", layer_indices=[target_idx])
                with torch.inference_mode():
                    ablated = model(**inputs, use_cache=False).logits[0, -1].float().cpu()
                condition = f"zero_ssm_layer_{target_idx}"

        delta_mean_abs = float(torch.mean(torch.abs(baseline - ablated)).item())
        delta_max_abs = float(torch.max(torch.abs(baseline - ablated)).item())

        result = {
            "model_key": model_key,
            "condition": condition,
            "delta_mean_abs": delta_mean_abs,
            "delta_max_abs": delta_max_abs,
            "baseline_top_tokens": top_tokens_from_logits(baseline, tokenizer),
            "ablated_top_tokens": top_tokens_from_logits(ablated, tokenizer),
        }
        ckpt.save(experiment_name, result)
        print(json.dumps(result, indent=2))
    finally:
        cleanup_model(model, tokenizer)

# Section 3 — Experiment 1: component ablation

The benchmark pipeline below is **manual by default** so it remains compatible with active hooks and patched forwards.
Each benchmark write is checkpointed separately, and the evaluation code itself also supports partial resume.


In [ ]:
# --- Cell 3A: Benchmark configuration, dataset loaders, and deterministic subset selection ---
# What this cell does:
# 1. Defines benchmark metadata
# 2. Provides robust dataset-loading wrappers
# 3. Creates deterministic subsets so every condition evaluates the same examples
#
# Expected output:
# - benchmark configuration DataFrame
# - loader helpers defined

NORMALIZED_RECORD_CACHE = {}

BENCHMARK_CONFIGS = {
    "mmlu": {
        "display_name": "MMLU",
        "metric_name": "accuracy",
        "split": "test",
        "shots": 5,
        "length_normalize": False,
    },
    "gsm8k": {
        "display_name": "GSM8K",
        "metric_name": "exact_match",
        "split": "test",
        "shots": 8,
        "length_normalize": False,
    },
    "arc_challenge": {
        "display_name": "ARC-Challenge",
        "metric_name": "accuracy",
        "split": "validation",
        "shots": 25,
        "length_normalize": False,
    },
    "hellaswag": {
        "display_name": "HellaSwag",
        "metric_name": "accuracy",
        "split": "validation",
        "shots": 10,
        "length_normalize": True,
    },
    "truthfulqa_mc": {
        "display_name": "TruthfulQA-MC",
        "metric_name": "accuracy",
        "split": "validation",
        "shots": 0,
        "length_normalize": True,
    },
}

def get_dataset_cached(path: str, name: str | None = None, split: str | None = None):
    key = (path, name, split)
    if key not in DATASET_CACHE:
        if name is None:
            DATASET_CACHE[key] = load_dataset(path, split=split)
        else:
            DATASET_CACHE[key] = load_dataset(path, name, split=split)
    return DATASET_CACHE[key]

def normalize_label_to_index(label):
    if isinstance(label, int):
        return label
    if isinstance(label, str):
        label = label.strip()
        if label.isdigit():
            return int(label)
        if label.upper() in LETTERS:
            return LETTERS.index(label.upper())
    raise ValueError(f"Could not normalize label: {label}")

def get_subset_indices(benchmark: str, total_len: int, max_samples: int | None, purpose: str):
    if max_samples is None or max_samples >= total_len:
        return list(range(total_len))

    subset_ckpt = f"subset_indices__{benchmark}__{purpose}__{max_samples}"
    cached = ckpt.load(subset_ckpt)
    if cached is not None:
        return cached["indices"]

    rng = np.random.default_rng(SEED)
    indices = np.arange(total_len)
    rng.shuffle(indices)
    selected = sorted(indices[:max_samples].tolist())

    payload = {
        "benchmark": benchmark,
        "purpose": purpose,
        "max_samples": max_samples,
        "indices": selected,
    }
    ckpt.save(subset_ckpt, payload)
    return selected

def normalize_mmlu_split(split: str):
    try:
        ds = get_dataset_cached("cais/mmlu", "all", split)
    except Exception:
        configs = [c for c in get_dataset_config_names("cais/mmlu") if c != "all"]
        parts = []
        for cfg_name in tqdm(configs, desc=f"Loading MMLU {split} subjects"):
            part = get_dataset_cached("cais/mmlu", cfg_name, split)
            if "subject" not in part.column_names:
                part = part.add_column("subject", [cfg_name] * len(part))
            parts.append(part)
        ds = concatenate_datasets(parts)

    records = []
    for i, ex in enumerate(ds):
        records.append({
            "id": f"{ex.get('subject', 'unknown')}__{i}",
            "subject": ex.get("subject", "unknown"),
            "question": ex["question"],
            "choices": list(ex["choices"]),
            "label": normalize_label_to_index(ex["answer"]),
        })
    return records

def normalize_arc_split(split: str):
    ds = get_dataset_cached("allenai/ai2_arc", "ARC-Challenge", split)
    records = []
    for i, ex in enumerate(ds):
        choices_text = list(ex["choices"]["text"])
        choices_label = list(ex["choices"]["label"])
        answer_key = str(ex["answerKey"]).strip()
        if answer_key in choices_label:
            label_idx = choices_label.index(answer_key)
        elif answer_key.isdigit():
            label_idx = int(answer_key) - 1
        else:
            continue
        records.append({
            "id": f"arc__{i}",
            "question": ex["question"],
            "choices": choices_text,
            "label": label_idx,
        })
    return records

def normalize_hellaswag_text(text: str) -> str:
    text = re.sub(r"\[.*?\]", "", text)
    text = text.replace("  ", " ")
    return text.strip()

def normalize_hellaswag_split(split: str):
    ds = get_dataset_cached("Rowan/hellaswag", None, split)
    records = []
    for i, ex in enumerate(ds):
        ctx = ex.get("ctx", "") or (str(ex.get("ctx_a", "")) + " " + str(ex.get("ctx_b", "")))
        records.append({
            "id": f"hellaswag__{int(ex['ind'])}",
            "context": normalize_hellaswag_text(ctx),
            "endings": [normalize_hellaswag_text(x) for x in ex["endings"]],
            "label": int(ex["label"]),
        })
    return records

def normalize_truthfulqa_split(split: str):
    ds = get_dataset_cached("rahmanidashti/truthful-qa", "multiple-choice", split)
    records = []
    for i, ex in enumerate(ds):
        # Flat format: "choices" + "label" keys present directly
        if "choices" in ex and "label" in ex:
            choices = list(ex["choices"])
            label = int(ex["label"])
        # mc1_targets format (EleutherAI / rahmanidashti canonical layout)
        elif "mc1_targets" in ex:
            choices = list(ex["mc1_targets"]["choices"])
            labels = list(ex["mc1_targets"]["labels"])
            label = int(np.argmax(labels))
        else:
            continue
        records.append({
            "id": f"truthfulqa__{i}",
            "question": ex["question"],
            "choices": choices,
            "label": label,
        })
    return records

def normalize_gsm8k_split(split: str):
    ds = get_dataset_cached("openai/gsm8k", "main", split)
    records = []
    for i, ex in enumerate(ds):
        records.append({
            "id": f"gsm8k__{i}",
            "question": ex["question"],
            "answer": ex["answer"],
        })
    return records

def get_benchmark_records(benchmark: str, max_samples: int | None = None, purpose: str = "default"):
    cache_key = (benchmark, BENCHMARK_CONFIGS[benchmark]["split"])
    if cache_key not in NORMALIZED_RECORD_CACHE:
        if benchmark == "mmlu":
            NORMALIZED_RECORD_CACHE[cache_key] = normalize_mmlu_split(BENCHMARK_CONFIGS[benchmark]["split"])
        elif benchmark == "arc_challenge":
            NORMALIZED_RECORD_CACHE[cache_key] = normalize_arc_split(BENCHMARK_CONFIGS[benchmark]["split"])
        elif benchmark == "hellaswag":
            NORMALIZED_RECORD_CACHE[cache_key] = normalize_hellaswag_split(BENCHMARK_CONFIGS[benchmark]["split"])
        elif benchmark == "truthfulqa_mc":
            NORMALIZED_RECORD_CACHE[cache_key] = normalize_truthfulqa_split(BENCHMARK_CONFIGS[benchmark]["split"])
        elif benchmark == "gsm8k":
            NORMALIZED_RECORD_CACHE[cache_key] = normalize_gsm8k_split(BENCHMARK_CONFIGS[benchmark]["split"])
        else:
            raise ValueError(f"Unknown benchmark: {benchmark}")

    records = NORMALIZED_RECORD_CACHE[cache_key]
    indices = get_subset_indices(benchmark, len(records), max_samples, purpose)
    return [records[i] for i in indices]

benchmark_df = pd.DataFrame.from_dict(BENCHMARK_CONFIGS, orient="index")
display(benchmark_df)

In [ ]:
# --- Cell 3B: Prompt builders, answer extraction, and log-likelihood scoring ---
# What this cell does:
# 1. Builds few-shot prompts for each benchmark
# 2. Scores multiple-choice candidates with batched log-likelihood
# 3. Adds GSM8K generation + answer extraction
#
# Expected output:
# - prompt / scorer helpers defined
# - no benchmark execution yet

def render_mc_question(question: str, choices: list[str], gold_idx: int | None = None) -> str:
    lines = [question.strip()]
    for i, choice in enumerate(choices):
        lines.append(f"{LETTERS[i]}. {str(choice).strip()}")
    lines.append("Answer:" if gold_idx is None else f"Answer: {LETTERS[gold_idx]}")
    return "\n".join(lines)

def get_mmlu_fewshot_by_subject():
    cache_name = "mmlu_fewshot_by_subject"
    cached = ckpt.load(cache_name)
    if cached is not None:
        return cached
    dev_records = normalize_mmlu_split("dev")
    by_subject = defaultdict(list)
    for ex in dev_records:
        by_subject[ex["subject"]].append(ex)
    ckpt.save(cache_name, dict(by_subject))
    return dict(by_subject)

def get_arc_fewshot_examples():
    cache_name = "arc_fewshot_examples"
    cached = ckpt.load(cache_name)
    if cached is not None:
        return cached
    records = normalize_arc_split("train")
    indices = get_subset_indices("arc_challenge_fewshot_train", len(records), BENCHMARK_CONFIGS["arc_challenge"]["shots"], "fewshot")
    selected = [records[i] for i in indices]
    ckpt.save(cache_name, selected)
    return selected

def get_hellaswag_fewshot_examples():
    cache_name = "hellaswag_fewshot_examples"
    cached = ckpt.load(cache_name)
    if cached is not None:
        return cached
    records = normalize_hellaswag_split("train")
    indices = get_subset_indices("hellaswag_fewshot_train", len(records), BENCHMARK_CONFIGS["hellaswag"]["shots"], "fewshot")
    selected = [records[i] for i in indices]
    ckpt.save(cache_name, selected)
    return selected

def get_gsm8k_fewshot_examples():
    cache_name = "gsm8k_fewshot_examples"
    cached = ckpt.load(cache_name)
    if cached is not None:
        return cached
    records = normalize_gsm8k_split("train")
    indices = get_subset_indices("gsm8k_fewshot_train", len(records), BENCHMARK_CONFIGS["gsm8k"]["shots"], "fewshot")
    selected = [records[i] for i in indices]
    ckpt.save(cache_name, selected)
    return selected

def build_mmlu_prompt(example):
    by_subject = get_mmlu_fewshot_by_subject()
    shots = by_subject[example["subject"]]
    blocks = ["The following are multiple choice questions (with answers)."]
    for shot in shots:
        blocks.append(render_mc_question(shot["question"], shot["choices"], shot["label"]))
    blocks.append(render_mc_question(example["question"], example["choices"], None))
    prompt = "\n\n".join(blocks)
    candidates = [f" {LETTERS[i]}" for i in range(len(example["choices"]))]
    return prompt, candidates, example["label"], False

def build_arc_prompt(example):
    shots = get_arc_fewshot_examples()
    blocks = ["Choose the correct answer option for each question."]
    for shot in shots:
        blocks.append(render_mc_question(shot["question"], shot["choices"], shot["label"]))
    blocks.append(render_mc_question(example["question"], example["choices"], None))
    prompt = "\n\n".join(blocks)
    candidates = [f" {LETTERS[i]}" for i in range(len(example["choices"]))]
    return prompt, candidates, example["label"], False

def build_hellaswag_prompt(example):
    shots = get_hellaswag_fewshot_examples()
    blocks = ["Complete each context with the most plausible ending."]
    for shot in shots:
        correct = shot["endings"][shot["label"]]
        blocks.append(f"Context: {shot['context']}\nEnding: {correct}")
    blocks.append(f"Context: {example['context']}\nEnding:")
    prompt = "\n\n".join(blocks)
    candidates = [f" {ending}" for ending in example["endings"]]
    return prompt, candidates, example["label"], True

def build_truthfulqa_prompt(example):
    prompt = (
        "Choose the most truthful answer to the question below.\n\n"
        f"Question: {example['question'].strip()}\n"
        "Answer:"
    )
    candidates = [f" {choice.strip()}" for choice in example["choices"]]
    return prompt, candidates, example["label"], True

def build_gsm8k_prompt(example):
    shots = get_gsm8k_fewshot_examples()
    blocks = ["Solve the following grade-school math problems. Show your reasoning briefly and end with `#### <answer>`."]  # no markdown parsing needed in prompt
    for shot in shots:
        blocks.append(f"Question: {shot['question'].strip()}\nAnswer: {shot['answer'].strip()}")
    blocks.append(f"Question: {example['question'].strip()}\nAnswer:")
    return "\n\n".join(blocks)

def truncate_prompt_ids(prompt_ids, continuation_ids, max_context: int):
    total = len(prompt_ids) + len(continuation_ids)
    if total <= max_context:
        return prompt_ids
    keep = max(1, max_context - len(continuation_ids) - 1)
    return prompt_ids[-keep:]

def score_candidate_continuations(
    model,
    tokenizer,
    prompt: str,
    candidates: list[str],
    length_normalize: bool = False,
    model_key: str = None,
):
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    seq_cap = get_max_inference_seq_length(model_key) if model_key else DEFAULT_MAX_INFERENCE_SEQ_LENGTH
    max_context = min(
        get_model_context_length(model),
        getattr(tokenizer, "model_max_length", get_model_context_length(model)),
        seq_cap,  # per-model cap to prevent Mamba-SSM quadratic OOM
    )

    raw_scores = []
    norm_scores = []

    # Process one candidate at a time to avoid OOM on models with large
    # intermediate activations (e.g. Falcon-H1 SSM quadratic expansion).
    for candidate in candidates:
        cand_ids = tokenizer(candidate, add_special_tokens=False)["input_ids"]
        trimmed_prompt_ids = truncate_prompt_ids(prompt_ids, cand_ids, max_context)

        full_ids = trimmed_prompt_ids + cand_ids
        p_len = len(trimmed_prompt_ids)
        c_len = len(cand_ids)

        input_ids = torch.tensor([full_ids], dtype=torch.long, device=DEVICE)
        attn_mask = torch.ones_like(input_ids, device=DEVICE)

        with torch.inference_mode():
            logits = model(
                input_ids=input_ids,
                attention_mask=attn_mask,
                use_cache=False
            ).logits

            log_probs = torch.log_softmax(logits[:, :-1, :], dim=-1)

        target_ids = input_ids[0, p_len:p_len + c_len]

        token_log_probs = log_probs[
            0,
            p_len - 1:p_len - 1 + c_len,
            :
        ].gather(-1, target_ids.unsqueeze(-1)).squeeze(-1)

        score = float(token_log_probs.sum().item())
        raw_scores.append(score)
        norm_scores.append(score / max(c_len, 1))

        del input_ids, attn_mask, logits, log_probs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return raw_scores, norm_scores

def normalize_number_string(text: str):
    text = text.strip()
    text = text.replace(",", "")
    m = re.search(r"-?\d+(?:\.\d+)?", text)
    return m.group(0) if m else None

def extract_gsm8k_final_answer(text: str):
    m = re.search(r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)", text)
    if m:
        return normalize_number_string(m.group(1))
    numbers = re.findall(r"-?\d+(?:\.\d+)?", text.replace(",", ""))
    return numbers[-1] if numbers else None

def generate_completion(model, tokenizer, prompt: str, max_new_tokens: int = 128, model_key: str = None):
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(DEVICE)
    # Truncate prompt to fit within the per-model seq cap (leaves room for generation)
    seq_cap = get_max_inference_seq_length(model_key) if model_key else DEFAULT_MAX_INFERENCE_SEQ_LENGTH
    max_input_len = max(1, seq_cap - max_new_tokens)
    if inputs["input_ids"].shape[1] > max_input_len:
        inputs["input_ids"] = inputs["input_ids"][:, -max_input_len:]
        inputs["attention_mask"] = inputs["attention_mask"][:, -max_input_len:]
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=False,  # safer when functional ablations are active
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

def benchmark_prompt_builder(benchmark: str, example: dict):
    if benchmark == "mmlu":
        return build_mmlu_prompt(example)
    if benchmark == "arc_challenge":
        return build_arc_prompt(example)
    if benchmark == "hellaswag":
        return build_hellaswag_prompt(example)
    if benchmark == "truthfulqa_mc":
        return build_truthfulqa_prompt(example)
    raise ValueError(f"Unsupported MC benchmark for prompt builder: {benchmark}")

In [ ]:
# --- Cell 3C: Resumable benchmark evaluators ---
# What this cell does:
# 1. Runs multiple-choice and GSM8K evaluation with partial resume
# 2. Saves incremental prediction lists to checkpoint after every few examples
# 3. Produces a consistent result schema across tasks
#
# Expected output:
# - evaluators defined
# - no benchmark execution yet

def summarize_binary_predictions(predictions: list[dict], metric_name: str = "accuracy"):
    if len(predictions) == 0:
        return {"n": 0, metric_name: None}
    values = [int(x["correct"]) for x in predictions]
    return {
        "n": len(values),
        metric_name: float(np.mean(values)),
        "correct_count": int(np.sum(values)),
    }

def evaluate_multiple_choice_benchmark(
    model,
    tokenizer,
    benchmark: str,
    records: list[dict],
    checkpoint_name: str,
    save_every: int = SAVE_EVERY_N_EXAMPLES,
    model_key: str = None,
):
    cached = ckpt.load(checkpoint_name)
    if cached is not None and cached.get("status") == "complete":
        ckpt.log(f"✅ {checkpoint_name} already completed. Loading from checkpoint.")
        return cached

    state = cached if cached is not None else {
        "status": "partial",
        "next_idx": 0,
        "predictions": [],
        "benchmark": benchmark,
    }

    predictions = list(state.get("predictions", []))
    start_idx = int(state.get("next_idx", len(predictions)))

    for idx in tqdm(range(start_idx, len(records)), desc=f"{benchmark}"):
        ex = records[idx]
        prompt, candidates, gold_idx, length_normalize = benchmark_prompt_builder(benchmark, ex)
        raw_scores, norm_scores = score_candidate_continuations(
            model,
            tokenizer,
            prompt,
            candidates,
            length_normalize=length_normalize,
            model_key=model_key,
        )
        scores = norm_scores if length_normalize else raw_scores
        pred_idx = int(np.argmax(scores))
        pred = {
            "example_id": ex["id"],
            "gold_idx": int(gold_idx),
            "pred_idx": pred_idx,
            "correct": int(pred_idx == gold_idx),
            "scores": [float(x) for x in scores],
            "raw_scores": [float(x) for x in raw_scores],
            "norm_scores": [float(x) for x in norm_scores],
        }
        predictions.append(pred)

        should_save = ((idx + 1) % save_every == 0) or (idx == len(records) - 1)
        if should_save:
            metric_name = BENCHMARK_CONFIGS[benchmark]["metric_name"]
            summary = summarize_binary_predictions(predictions, metric_name=metric_name)
            payload = {
                "status": "complete" if idx == len(records) - 1 else "partial",
                "next_idx": idx + 1,
                "benchmark": benchmark,
                "predictions": predictions,
                "summary": summary,
            }
            ckpt.save(checkpoint_name, payload)

    return ckpt.load(checkpoint_name)

def evaluate_gsm8k_benchmark(
    model,
    tokenizer,
    records: list[dict],
    checkpoint_name: str,
    save_every: int = max(2, SAVE_EVERY_N_EXAMPLES // 2),
    max_new_tokens: int = 128,
    model_key: str = None,
):
    cached = ckpt.load(checkpoint_name)
    if cached is not None and cached.get("status") == "complete":
        ckpt.log(f"✅ {checkpoint_name} already completed. Loading from checkpoint.")
        return cached

    state = cached if cached is not None else {
        "status": "partial",
        "next_idx": 0,
        "predictions": [],
        "benchmark": "gsm8k",
    }

    predictions = list(state.get("predictions", []))
    start_idx = int(state.get("next_idx", len(predictions)))

    for idx in tqdm(range(start_idx, len(records)), desc="gsm8k"):
        ex = records[idx]
        prompt = build_gsm8k_prompt(ex)
        completion = generate_completion(model, tokenizer, prompt, max_new_tokens=max_new_tokens, model_key=model_key)
        pred_answer = extract_gsm8k_final_answer(completion)
        gold_answer = extract_gsm8k_final_answer(ex["answer"])

        pred = {
            "example_id": ex["id"],
            "gold_answer": gold_answer,
            "pred_answer": pred_answer,
            "completion": completion,
            "correct": int(pred_answer is not None and gold_answer is not None and pred_answer == gold_answer),
        }
        predictions.append(pred)

        should_save = ((idx + 1) % save_every == 0) or (idx == len(records) - 1)
        if should_save:
            summary = summarize_binary_predictions(predictions, metric_name="exact_match")
            payload = {
                "status": "complete" if idx == len(records) - 1 else "partial",
                "next_idx": idx + 1,
                "benchmark": "gsm8k",
                "predictions": predictions,
                "summary": summary,
            }
            ckpt.save(checkpoint_name, payload)

    return ckpt.load(checkpoint_name)

def evaluate_condition(
    model,
    tokenizer,
    model_key: str,
    condition_name: str,
    benchmarks: list[str],
    max_samples_main: int | None,
    sample_purpose: str,
):
    results = {}
    for benchmark in benchmarks:
        if benchmark == "gsm8k":
            max_samples = MAX_GSM8K_SAMPLES if max_samples_main is not None else None
        else:
            max_samples = max_samples_main

        records = get_benchmark_records(
            benchmark,
            max_samples=max_samples,
            purpose=f"{sample_purpose}__{benchmark}",
        )

        ckpt_name = f"eval__{model_key}__{condition_name}__{benchmark}"
        if benchmark == "gsm8k":
            result = evaluate_gsm8k_benchmark(model, tokenizer, records, ckpt_name, model_key=model_key)
        else:
            result = evaluate_multiple_choice_benchmark(model, tokenizer, benchmark, records, ckpt_name, model_key=model_key)

        results[benchmark] = result
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

    return results

In [ ]:
# --- Cell 3D: Experiment 1 runner (baseline, group ablations, individual layers, progressive / positional) ---
# What this cell does:
# 1. Runs the core ablation experiment for each model
# 2. Saves checkpoints after every benchmark and every layer
# 3. Includes Qwen progressive ablation and Falcon positional ablation
#
# Expected output:
# - many checkpoint messages
# - a nested results dictionary in memory

def chunk_indices(indices: list[int], num_chunks: int):
    if len(indices) == 0:
        return []
    chunk_size = math.ceil(len(indices) / num_chunks)
    return [indices[i:i + chunk_size] for i in range(0, len(indices), chunk_size)]

def run_experiment1_for_model(model_key: str):
    experiment_name = f"exp1_master__{model_key}"
    cached = ckpt.load(experiment_name)
    if cached is not None and cached.get("status") == "complete":
        ckpt.log(f"✅ {experiment_name} already completed. Loading from checkpoint.")
        return cached

    arch = get_architecture_result(model_key)["summary"]
    model, tokenizer, meta = load_model_and_tokenizer(model_key)

    results = {
        "model_key": model_key,
        "meta": meta,
        "conditions": {},
    }

    try:
        # -----------------------
        # Baseline
        # -----------------------
        baseline_name = "baseline"
        results["conditions"][baseline_name] = evaluate_condition(
            model,
            tokenizer,
            model_key,
            baseline_name,
            BENCHMARKS_MAIN,
            max_samples_main=MAX_EVAL_SAMPLES,
            sample_purpose=f"{model_key}__main_eval",
        )

        arch_family = MODEL_SPECS[model_key]["arch_family"]

        # -----------------------
        # Group ablations
        # -----------------------
        if arch_family == "qwen_sequential_hybrid":
            with AblationManager(model, model_key, arch) as ablator:
                ablator.skip_all_layers_of_type("linear")
                condition_name = "all_linear_off"
                results["conditions"][condition_name] = evaluate_condition(
                    model,
                    tokenizer,
                    model_key,
                    condition_name,
                    BENCHMARKS_MAIN,
                    max_samples_main=MAX_EVAL_SAMPLES,
                    sample_purpose=f"{model_key}__main_eval",
                )

            with AblationManager(model, model_key, arch) as ablator:
                ablator.skip_all_layers_of_type("attention")
                condition_name = "all_attention_off"
                results["conditions"][condition_name] = evaluate_condition(
                    model,
                    tokenizer,
                    model_key,
                    condition_name,
                    BENCHMARKS_MAIN,
                    max_samples_main=MAX_EVAL_SAMPLES,
                    sample_purpose=f"{model_key}__main_eval",
                )

            # Matched random controls for Qwen group ablation
            num_layers_total = arch["num_layers"]
            linear_indices = list(arch["linear_indices"])
            attention_indices = [i for i, t in enumerate(arch["layer_types_norm"]) if t == "attention"]
            num_linear = len(linear_indices)
            num_attention = len(attention_indices)
            rng_ctrl = np.random.default_rng(SEED + 1)

            for trial in range(3):
                # Random control matching all_linear_off count
                condition_name = f"random_control_linear_trial{trial}"
                if condition_name not in results["conditions"]:
                    sampled = sorted(rng_ctrl.choice(num_layers_total, size=min(num_linear, num_layers_total), replace=False).tolist())
                    with AblationManager(model, model_key, arch) as ablator:
                        for idx in sampled:
                            ablator.skip_layer(idx)
                        results["conditions"][condition_name] = evaluate_condition(
                            model, tokenizer, model_key, condition_name,
                            BENCHMARKS_MAIN, max_samples_main=MAX_EVAL_SAMPLES,
                            sample_purpose=f"{model_key}__main_eval",
                        )
                    ckpt.save(experiment_name, {"status": "partial", "payload": results})

                # Random control matching all_attention_off count
                condition_name = f"random_control_attention_trial{trial}"
                if condition_name not in results["conditions"]:
                    sampled = sorted(rng_ctrl.choice(num_layers_total, size=min(num_attention, num_layers_total), replace=False).tolist())
                    with AblationManager(model, model_key, arch) as ablator:
                        for idx in sampled:
                            ablator.skip_layer(idx)
                        results["conditions"][condition_name] = evaluate_condition(
                            model, tokenizer, model_key, condition_name,
                            BENCHMARKS_MAIN, max_samples_main=MAX_EVAL_SAMPLES,
                            sample_purpose=f"{model_key}__main_eval",
                        )
                    ckpt.save(experiment_name, {"status": "partial", "payload": results})

        else:
            with AblationManager(model, model_key, arch) as ablator:
                ablator.zero_component_output("ssm")
                condition_name = "all_ssm_off"
                results["conditions"][condition_name] = evaluate_condition(
                    model,
                    tokenizer,
                    model_key,
                    condition_name,
                    BENCHMARKS_MAIN,
                    max_samples_main=MAX_EVAL_SAMPLES,
                    sample_purpose=f"{model_key}__main_eval",
                )

            with AblationManager(model, model_key, arch) as ablator:
                ablator.zero_component_output("attention")
                condition_name = "all_attention_off"
                results["conditions"][condition_name] = evaluate_condition(
                    model,
                    tokenizer,
                    model_key,
                    condition_name,
                    BENCHMARKS_MAIN,
                    max_samples_main=MAX_EVAL_SAMPLES,
                    sample_purpose=f"{model_key}__main_eval",
                )

            # Matched random controls for Falcon group ablation
            num_layers_falcon = arch["num_layers"]
            all_component_blocks = []
            for comp in ["attention", "ssm"]:
                for blk_idx in range(num_layers_falcon):
                    all_component_blocks.append((comp, blk_idx))
            num_ssm_blocks = num_layers_falcon  # number of SSM component-blocks zeroed in all_ssm_off
            rng_ctrl = np.random.default_rng(SEED + 1)

            for trial in range(3):
                condition_name = f"random_control_ssm_trial{trial}"
                if condition_name not in results["conditions"]:
                    sampled_pairs = [all_component_blocks[i] for i in sorted(
                        rng_ctrl.choice(len(all_component_blocks), size=min(num_ssm_blocks, len(all_component_blocks)), replace=False).tolist()
                    )]
                    with AblationManager(model, model_key, arch) as ablator:
                        for comp, blk_idx in sampled_pairs:
                            ablator.zero_component_output(comp, layer_indices=[blk_idx])
                        results["conditions"][condition_name] = evaluate_condition(
                            model, tokenizer, model_key, condition_name,
                            BENCHMARKS_MAIN, max_samples_main=MAX_EVAL_SAMPLES,
                            sample_purpose=f"{model_key}__main_eval",
                        )
                    ckpt.save(experiment_name, {"status": "partial", "payload": results})

        ckpt.save(experiment_name, {"status": "partial", "payload": results})

        # -----------------------
        # Individual layer / block ablations
        # -----------------------
        if arch_family == "qwen_sequential_hybrid":
            num_layers = arch["num_layers"]
            for layer_idx in range(num_layers):
                layer_type = arch["layer_types_norm"][layer_idx]
                condition_name = f"layer_{layer_idx:02d}_{layer_type}_off"
                if condition_name in results["conditions"]:
                    continue
                with AblationManager(model, model_key, arch) as ablator:
                    ablator.skip_layer(layer_idx)
                    results["conditions"][condition_name] = evaluate_condition(
                        model,
                        tokenizer,
                        model_key,
                        condition_name,
                        LAYER_SWEEP_BENCHMARKS,
                        max_samples_main=MAX_LAYER_ABLATION_SAMPLES,
                        sample_purpose=f"{model_key}__layer_eval",
                    )
                ckpt.save(experiment_name, {"status": "partial", "payload": results})

            # Progressive linear ablations: first N, last N, random N
            linear_indices = list(arch["linear_indices"])
            rng = np.random.default_rng(SEED)
            for n in PROGRESSIVE_ABLATION_STEPS:
                n = min(n, len(linear_indices))
                subsets = {
                    f"progressive_linear_first_{n:02d}": linear_indices[:n],
                    f"progressive_linear_last_{n:02d}": linear_indices[-n:],
                    f"progressive_linear_random_{n:02d}": sorted(rng.choice(linear_indices, size=n, replace=False).tolist()),
                }
                for condition_name, layer_subset in subsets.items():
                    if condition_name in results["conditions"]:
                        continue
                    with AblationManager(model, model_key, arch) as ablator:
                        for idx in layer_subset:
                            ablator.skip_layer(idx)
                        results["conditions"][condition_name] = evaluate_condition(
                            model,
                            tokenizer,
                            model_key,
                            condition_name,
                            PROGRESSIVE_ABLATION_BENCHMARKS,
                            max_samples_main=MAX_PROGRESSIVE_ABLATION_SAMPLES,
                            sample_purpose=f"{model_key}__progressive_eval",
                        )
                    ckpt.save(experiment_name, {"status": "partial", "payload": results})

        else:
            num_layers = arch["num_layers"]
            for component_type in ["attention", "ssm"]:
                for layer_idx in range(num_layers):
                    condition_name = f"{component_type}_layer_{layer_idx:02d}_off"
                    if condition_name in results["conditions"]:
                        continue
                    with AblationManager(model, model_key, arch) as ablator:
                        ablator.zero_component_output(component_type, layer_indices=[layer_idx])
                        results["conditions"][condition_name] = evaluate_condition(
                            model,
                            tokenizer,
                            model_key,
                            condition_name,
                            LAYER_SWEEP_BENCHMARKS,
                            max_samples_main=MAX_LAYER_ABLATION_SAMPLES,
                            sample_purpose=f"{model_key}__layer_eval",
                        )
                    ckpt.save(experiment_name, {"status": "partial", "payload": results})

            # Positional ablations: early / middle / late for each component
            layer_buckets = chunk_indices(list(range(num_layers)), FALCON_POSITION_BUCKETS)
            bucket_names = ["early", "middle", "late"][:len(layer_buckets)]
            for component_type in ["attention", "ssm"]:
                for bucket_name, bucket_indices in zip(bucket_names, layer_buckets):
                    condition_name = f"{component_type}_{bucket_name}_off"
                    if condition_name in results["conditions"]:
                        continue
                    with AblationManager(model, model_key, arch) as ablator:
                        ablator.zero_component_output(component_type, layer_indices=bucket_indices)
                        results["conditions"][condition_name] = evaluate_condition(
                            model,
                            tokenizer,
                            model_key,
                            condition_name,
                            BENCHMARKS_MAIN,
                            max_samples_main=MAX_EVAL_SAMPLES,
                            sample_purpose=f"{model_key}__main_eval",
                        )
                    ckpt.save(experiment_name, {"status": "partial", "payload": results})

        final_payload = {"status": "complete", "payload": results}
        ckpt.save(experiment_name, final_payload)
        return final_payload
    finally:
        cleanup_model(model, tokenizer)

In [ ]:
# --- Cell 3E: Execute Experiment 1 for all selected models ---
# What this cell does:
# 1. Runs the full ablation study for each model sequentially
# 2. Stores the nested result structure in `exp1_results`
#
# Expected output:
# - a long stream of checkpoint logs
# - `exp1_results` populated

exp1_results = {}
for model_key in MODELS_TO_RUN:
    try:
        exp1_results[model_key] = run_experiment1_for_model(model_key)
    except Exception as exc:
        ckpt.log(f"[ERROR] Experiment 1 failed for {model_key}: {exc}")
        traceback.print_exc()
        exp1_results[model_key] = {"status": "failed", "error": str(exc)}

print("Experiment 1 run complete.")

if "exp1_results" not in globals() or not isinstance(exp1_results, dict) or len(exp1_results) == 0:
    exp1_results = {model_key: ckpt.load(f"exp1_master__{model_key}") for model_key in MODELS_TO_RUN}

In [ ]:
# --- Cell 3F: Collect Experiment 1 summaries and generate plots / tables ---
# What this cell does:
# 1. Aggregates baseline and ablation results into tidy tables
# 2. Builds heatmaps and bar charts
# 3. Saves CSV + LaTeX + figure artifacts to Drive
#
# Expected output:
# - summary DataFrame
# - multiple files written to Drive

def get_condition_summary(result_obj):
    payload = result_obj["payload"] if isinstance(result_obj, dict) and "payload" in result_obj else result_obj
    if payload is None:
        return []
    rows = []
    for condition_name, benchmark_results in payload["conditions"].items():
        for benchmark, result in benchmark_results.items():
            summary = result["summary"]
            metric_name = BENCHMARK_CONFIGS[benchmark]["metric_name"]
            rows.append({
                "model_key": payload["model_key"],
                "condition": condition_name,
                "benchmark": benchmark,
                "metric_name": metric_name,
                "score": summary[metric_name],
                "n": summary["n"],
            })
    return rows

exp1_rows = []
for model_key, result in exp1_results.items():
    if isinstance(result, dict) and result.get("status") == "failed":
        continue
    exp1_rows.extend(get_condition_summary(result))

exp1_df = pd.DataFrame(exp1_rows)
save_dataframe(exp1_df, "experiment1_summary", index=False)
display(exp1_df.head(20))

# Main table: benchmark x condition x model
pivot_df = exp1_df.pivot_table(index=["model_key", "condition"], columns="benchmark", values="score")
save_dataframe(pivot_df.reset_index(), "experiment1_pivot", index=False)
display(pivot_df.head(20))

# Group-ablation bar chart (including matched random controls)
group_conditions = ["all_linear_off", "all_attention_off", "all_ssm_off"]
random_control_prefixes = ["random_control_linear_trial", "random_control_attention_trial", "random_control_ssm_trial"]

# Compute averaged random control conditions
random_ctrl_rows = []
for model_key in MODELS_TO_RUN:
    model_df_tmp = exp1_df[exp1_df["model_key"] == model_key]
    for prefix in random_control_prefixes:
        trial_rows = model_df_tmp[model_df_tmp["condition"].str.startswith(prefix)]
        if len(trial_rows) == 0:
            continue
        avg = trial_rows.groupby("benchmark").agg(score_mean=("score", "mean"), score_std=("score", "std"), n=("n", "first")).reset_index()
        ctrl_name = prefix.replace("_trial", "")  # e.g. random_control_linear
        for _, row in avg.iterrows():
            random_ctrl_rows.append({
                "model_key": model_key,
                "condition": ctrl_name,
                "benchmark": row["benchmark"],
                "score": row["score_mean"],
                "score_std": row["score_std"],
                "n": row["n"],
            })

all_conditions = group_conditions + [p.replace("_trial", "") for p in random_control_prefixes]
bar_df = exp1_df[exp1_df["condition"].isin(group_conditions + ["baseline"])].copy()
if random_ctrl_rows:
    bar_df = pd.concat([bar_df, pd.DataFrame(random_ctrl_rows)], ignore_index=True)

if len(bar_df) > 0:
    fig, ax = plt.subplots(figsize=(14, 5))
    plot_rows = []
    for model_key in sorted(bar_df["model_key"].unique()):
        model_subset = bar_df[bar_df["model_key"] == model_key]
        baseline_scores = model_subset[model_subset["condition"] == "baseline"].set_index("benchmark")["score"].to_dict()
        for condition in [c for c in all_conditions if c in model_subset["condition"].unique()]:
            cond_subset = model_subset[model_subset["condition"] == condition]
            cond_scores = cond_subset.set_index("benchmark")["score"].to_dict()
            common = sorted(set(baseline_scores) & set(cond_scores))
            if not common:
                continue
            degradation = np.mean([baseline_scores[b] - cond_scores[b] for b in common])
            # Compute error bar from std across trials if available
            if "score_std" in cond_subset.columns and cond_subset["score_std"].notna().any():
                err = np.mean([cond_subset.set_index("benchmark")["score_std"].to_dict().get(b, 0) for b in common])
            else:
                err = 0
            plot_rows.append({"model_key": model_key, "condition": condition, "mean_degradation": degradation, "err": err})

    plot_df = pd.DataFrame(plot_rows)
    if len(plot_df) > 0:
        conditions_present = sorted(plot_df["condition"].unique())
        n_conditions = len(conditions_present)
        bar_width = 0.8 / max(n_conditions, 1)
        for i, condition in enumerate(conditions_present):
            sub = plot_df[plot_df["condition"] == condition]
            x = np.arange(len(sub))
            is_random = condition.startswith("random_control")
            ax.bar(x + i * bar_width, sub["mean_degradation"], width=bar_width, label=condition,
                   yerr=sub["err"] if is_random else None,
                   alpha=0.5 if is_random else 1.0,
                   hatch="//" if is_random else None)
        ax.set_xticks(np.arange(len(sub)) + bar_width * (n_conditions - 1) / 2)
        ax.set_xticklabels(list(sub["model_key"]))
        ax.set_ylabel("Mean score drop vs baseline")
        ax.set_title("Experiment 1: mean degradation under group ablation (with random controls)")
        ax.legend(fontsize=8)
        save_figure(fig, "experiment1_group_ablation_bar")

# Heatmaps for layer / block ablations
for model_key in MODELS_TO_RUN:
    model_df = exp1_df[exp1_df["model_key"] == model_key].copy()
    if len(model_df) == 0:
        continue

    baseline_scores = model_df[model_df["condition"] == "baseline"].set_index("benchmark")["score"].to_dict()

    if MODEL_SPECS[model_key]["arch_family"] == "qwen_sequential_hybrid":
        layer_rows = model_df[model_df["condition"].str.startswith("layer_")].copy()
        if len(layer_rows) > 0:
            layer_rows["layer_idx"] = layer_rows["condition"].str.extract(r"layer_(\d+)").astype(int)
            layer_rows["degradation"] = layer_rows.apply(lambda r: baseline_scores.get(r["benchmark"], np.nan) - r["score"], axis=1)
            heatmap = layer_rows.pivot_table(index="benchmark", columns="layer_idx", values="degradation")
            fig, ax = plt.subplots(figsize=(12, 4))
            im = ax.imshow(heatmap.values, aspect="auto")
            ax.set_xticks(np.arange(len(heatmap.columns)))
            ax.set_xticklabels(list(heatmap.columns))
            ax.set_yticks(np.arange(len(heatmap.index)))
            ax.set_yticklabels(list(heatmap.index))
            ax.set_xlabel("Layer index")
            ax.set_title(f"{model_key}: degradation heatmap for individual layer ablations")
            fig.colorbar(im, ax=ax)
            save_figure(fig, f"{model_key}_layer_ablation_heatmap")
    else:
        for component_type in ["attention", "ssm"]:
            layer_rows = model_df[model_df["condition"].str.startswith(f"{component_type}_layer_")].copy()
            if len(layer_rows) == 0:
                continue
            layer_rows["layer_idx"] = layer_rows["condition"].str.extract(rf"{component_type}_layer_(\d+)").astype(int)
            layer_rows["degradation"] = layer_rows.apply(lambda r: baseline_scores.get(r["benchmark"], np.nan) - r["score"], axis=1)
            heatmap = layer_rows.pivot_table(index="benchmark", columns="layer_idx", values="degradation")
            fig, ax = plt.subplots(figsize=(12, 4))
            im = ax.imshow(heatmap.values, aspect="auto")
            ax.set_xticks(np.arange(len(heatmap.columns)))
            ax.set_xticklabels(list(heatmap.columns))
            ax.set_yticks(np.arange(len(heatmap.index)))
            ax.set_yticklabels(list(heatmap.index))
            ax.set_xlabel("Layer index")
            ax.set_title(f"{model_key}: degradation heatmap for {component_type} block ablations")
            fig.colorbar(im, ax=ax)
            save_figure(fig, f"{model_key}_{component_type}_ablation_heatmap")

print("Experiment 1 aggregation complete.")

In [ ]:
# --- Cell 3F-extra: Additional publication figures for Exp 1 ---
# What this cell does:
# 1. Radar chart: baseline vs group ablations across all benchmarks (both models)
# 2. Side-by-side dual heatmap: layer sweep comparison Qwen vs Falcon
# 3. Random control significance: actual ablation vs random controls with error bars
# 4. Component dominance by layer position
# 5. Compact Table 1 (paper-ready) with delta from baseline
#
# Expected output:
# - 5 additional publication-quality figures

import seaborn as sns
sns.set_style("whitegrid")
sns.set_context("paper", font_scale=1.2)

# ── FIGURE A: Radar chart — baseline vs group ablations ──────────────
def radar_chart(df, model_key, ax, title):
    benchmarks = sorted(df["benchmark"].unique())
    N = len(benchmarks)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]  # close the polygon

    baseline = df[df["condition"] == "baseline"].set_index("benchmark")["score"]
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([BENCHMARK_CONFIGS[b]["display_name"] for b in benchmarks], size=8)
    ax.set_ylim(0, 1)
    ax.set_title(title, size=11, pad=15)

    colors = {"baseline": "#2196F3", "all_linear_off": "#FF9800", "all_attention_off": "#F44336",
              "all_ssm_off": "#FF9800"}
    conditions_to_plot = ["baseline"] + [c for c in df["condition"].unique()
                                         if c in ["all_linear_off", "all_attention_off", "all_ssm_off"]]

    for condition in conditions_to_plot:
        sub = df[df["condition"] == condition].set_index("benchmark")["score"]
        values = [sub.get(b, 0) for b in benchmarks] + [sub.get(benchmarks[0], 0)]
        color = colors.get(condition, "#9E9E9E")
        ax.plot(angles, values, "o-", linewidth=1.5, label=condition.replace("_", " "), color=color)
        ax.fill(angles, values, alpha=0.1, color=color)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=7)

try:
    group_df = exp1_df[exp1_df["condition"].isin(["baseline", "all_linear_off", "all_attention_off", "all_ssm_off"])]
    models = sorted(group_df["model_key"].unique())
    fig, axes = plt.subplots(1, len(models), figsize=(6 * len(models), 5), subplot_kw=dict(polar=True))
    if len(models) == 1:
        axes = [axes]
    for ax, mk in zip(axes, models):
        radar_chart(group_df[group_df["model_key"] == mk], mk, ax,
                    MODEL_SPECS[mk]["display_name"])
    fig.suptitle("Benchmark performance: baseline vs group ablations", y=1.02, fontsize=13)
    plt.tight_layout()
    save_figure(fig, "paper_radar_group_ablation")
    print("✅ Radar chart saved")
except Exception as e:
    print(f"Radar chart skipped: {e}")


# ── FIGURE B: Side-by-side layer sweep heatmaps ─────────────────────
try:
    layer_conditions_attn = sorted([c for c in exp1_df["condition"].unique() if "attention_layer_" in c or "layer_" in c and "attention" in c])
    layer_conditions_ssm = sorted([c for c in exp1_df["condition"].unique() if "ssm_layer_" in c])
    layer_conditions_linear = sorted([c for c in exp1_df["condition"].unique() if "linear" in c and "layer_" in c])

    # Build delta-from-baseline matrices for each model
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)

    for ax_idx, model_key in enumerate(sorted(exp1_df["model_key"].unique())):
        model_df = exp1_df[exp1_df["model_key"] == model_key]
        baseline_scores = model_df[model_df["condition"] == "baseline"].set_index("benchmark")["score"]

        # Find layer conditions for this model
        layer_conds = sorted([c for c in model_df["condition"].unique()
                              if ("layer_" in c and "_off" in c and "random" not in c
                                  and c not in ["all_linear_off", "all_attention_off", "all_ssm_off"])])

        if not layer_conds:
            axes[ax_idx].text(0.5, 0.5, "No layer sweep data", ha="center", va="center")
            axes[ax_idx].set_title(MODEL_SPECS[model_key]["display_name"])
            continue

        benchmarks_available = sorted(model_df[model_df["condition"].isin(layer_conds)]["benchmark"].unique())
        delta_matrix = []
        ylabels = []
        for cond in layer_conds:
            cond_scores = model_df[model_df["condition"] == cond].set_index("benchmark")["score"]
            row = []
            for b in benchmarks_available:
                base = baseline_scores.get(b, 0)
                ablated = cond_scores.get(b, 0)
                row.append(base - ablated)  # positive = degradation
            delta_matrix.append(row)
            # Extract layer number for label
            parts = cond.replace("_off", "").split("_layer_")
            if len(parts) == 2:
                ylabels.append(f"L{parts[1]} ({parts[0][:3]})")
            else:
                ylabels.append(cond[:20])

        delta_arr = np.array(delta_matrix)
        im = axes[ax_idx].imshow(delta_arr, aspect="auto", cmap="RdYlGn_r", vmin=0,
                                  vmax=max(0.3, np.percentile(delta_arr, 95)))
        axes[ax_idx].set_xticks(range(len(benchmarks_available)))
        axes[ax_idx].set_xticklabels([BENCHMARK_CONFIGS[b]["display_name"] for b in benchmarks_available],
                                      rotation=30, ha="right", fontsize=8)
        axes[ax_idx].set_yticks(range(len(ylabels)))
        axes[ax_idx].set_yticklabels(ylabels, fontsize=6)
        axes[ax_idx].set_title(MODEL_SPECS[model_key]["display_name"], fontsize=11)

    fig.suptitle("Layer-wise ablation impact (Δ accuracy from baseline)", fontsize=13)
    fig.colorbar(im, ax=axes, shrink=0.8, label="Accuracy drop")
    plt.tight_layout()
    save_figure(fig, "paper_dual_layer_sweep_heatmap")
    print("✅ Dual layer sweep heatmap saved")
except Exception as e:
    print(f"Dual heatmap skipped: {e}")


# ── FIGURE C: Random control significance ────────────────────────────
try:
    group_conds = ["all_linear_off", "all_attention_off", "all_ssm_off"]
    random_prefixes = ["random_control_linear", "random_control_attention", "random_control_ssm"]

    fig, axes = plt.subplots(1, len(MODELS_TO_RUN), figsize=(7 * len(MODELS_TO_RUN), 5), squeeze=False)

    for col_idx, model_key in enumerate(sorted(MODELS_TO_RUN)):
        ax = axes[0, col_idx]
        model_df = exp1_df[exp1_df["model_key"] == model_key]
        baseline = model_df[model_df["condition"] == "baseline"]
        if len(baseline) == 0:
            continue
        base_mean = baseline.groupby("benchmark")["score"].mean()

        # Average score across benchmarks for each condition
        bars_data = []

        # Baseline
        bars_data.append({"label": "baseline", "mean": base_mean.mean(), "std": 0, "color": "#2196F3"})

        for gc, rp in zip(group_conds, random_prefixes):
            # Actual ablation
            gc_df = model_df[model_df["condition"] == gc]
            if len(gc_df) > 0:
                gc_scores = gc_df.groupby("benchmark")["score"].mean()
                bars_data.append({"label": gc.replace("all_", "").replace("_off", ""),
                                  "mean": gc_scores.mean(), "std": 0, "color": "#F44336"})

            # Random controls (trials)
            trial_conds = [c for c in model_df["condition"].unique() if c.startswith(rp)]
            if trial_conds:
                trial_means = []
                for tc in trial_conds:
                    tc_scores = model_df[model_df["condition"] == tc].groupby("benchmark")["score"].mean()
                    trial_means.append(tc_scores.mean())
                bars_data.append({"label": f"random\n(n={len(trial_conds)})",

                                  "mean": np.mean(trial_means), "std": np.std(trial_means),
                                  "color": "#9E9E9E"})

        if bars_data:
            x = range(len(bars_data))
            ax.bar(x, [d["mean"] for d in bars_data],
                   yerr=[d["std"] for d in bars_data],
                   color=[d["color"] for d in bars_data],
                   capsize=4, edgecolor="black", linewidth=0.5)
            ax.set_xticks(x)
            ax.set_xticklabels([d["label"] for d in bars_data], fontsize=8)
            ax.set_ylabel("Mean accuracy (all benchmarks)")
            ax.set_title(MODEL_SPECS[model_key]["display_name"])
            ax.set_ylim(0, max(d["mean"] for d in bars_data) * 1.15)

    fig.suptitle("Targeted ablation vs random layer removal", fontsize=13)
    plt.tight_layout()
    save_figure(fig, "paper_random_control_significance")
    print("✅ Random control significance plot saved")
except Exception as e:
    print(f"Random control plot skipped: {e}")


# ── FIGURE D: Component dominance by layer position ──────────────────
try:
    fig, axes = plt.subplots(1, len(MODELS_TO_RUN), figsize=(7 * len(MODELS_TO_RUN), 4.5), squeeze=False)

    for col_idx, model_key in enumerate(sorted(MODELS_TO_RUN)):
        ax = axes[0, col_idx]
        model_df = exp1_df[exp1_df["model_key"] == model_key]
        baseline = model_df[model_df["condition"] == "baseline"]
        if len(baseline) == 0:
            continue
        base_scores = baseline.set_index("benchmark")["score"]

        # Separate component types
        arch_family = MODEL_SPECS[model_key]["arch_family"]
        if arch_family == "qwen_sequential_hybrid":
            comp_a, comp_b = "linear", "attention"
            label_a, label_b = "Linear attention", "Full attention"
        else:
            comp_a, comp_b = "ssm", "attention"
            label_a, label_b = "SSM (Mamba)", "Attention"


        layer_conds_a = sorted([c for c in model_df["condition"].unique()
                                if (c.startswith(f"{comp_a}_layer_") or f"_{comp_a}_off" in c)
                                and c.endswith("_off") and "all_" not in c and "random" not in c
                                and "early" not in c and "middle" not in c and "late" not in c])
        layer_conds_b = sorted([c for c in model_df["condition"].unique()
                                if (c.startswith(f"{comp_b}_layer_") or f"_{comp_b}_off" in c)
                                and c.endswith("_off") and "all_" not in c and "random" not in c
                                and "early" not in c and "middle" not in c and "late" not in c])

        def avg_drop(conds):
            drops = []
            for c in conds:
                c_scores = model_df[model_df["condition"] == c].set_index("benchmark")["score"]
                common = set(base_scores.index) & set(c_scores.index)
                if common:
                    drop = np.mean([base_scores[b] - c_scores[b] for b in common])
                    drops.append(drop)
                else:
                    drops.append(0)
            return drops

        drops_a = avg_drop(layer_conds_a)
        drops_b = avg_drop(layer_conds_b)

        # layers_a = list(range(len(drops_a)))
        # layers_b = list(range(len(drops_b)))
        import re
        def extract_layer_idx(cond):
            m = re.search(r'(\d+)', cond)
            return int(m.group(1)) if m else 0
        layers_a = [extract_layer_idx(c) for c in layer_conds_a]
        layers_b = [extract_layer_idx(c) for c in layer_conds_b]
        ax.fill_between(layers_a, drops_a, alpha=0.3, color="#FF9800", label=label_a)
        ax.plot(layers_a, drops_a, "o-", color="#FF9800", markersize=3, linewidth=1.2)
        ax.fill_between(layers_b, drops_b, alpha=0.3, color="#2196F3", label=label_b)
        ax.plot(layers_b, drops_b, "s-", color="#2196F3", markersize=3, linewidth=1.2)

        ax.set_xlabel("Layer index")
        ax.set_ylabel("Mean accuracy drop")
        ax.set_title(MODEL_SPECS[model_key]["display_name"])
        ax.legend(fontsize=9)
        ax.axhline(y=0, color="gray", linestyle="--", linewidth=0.5)

    fig.suptitle("Component importance by layer position", fontsize=13)
    plt.tight_layout()
    save_figure(fig, "paper_component_dominance_by_layer")
    print("✅ Component dominance plot saved")
except Exception as e:
    print(f"Component dominance plot skipped: {e}")


# ── FIGURE E: Compact delta table (paper Table 1) ───────────────────
try:
    table_rows = []
    for model_key in sorted(exp1_df["model_key"].unique()):
        model_df = exp1_df[exp1_df["model_key"] == model_key]
        baseline = model_df[model_df["condition"] == "baseline"].set_index("benchmark")["score"]
        group_conds_present = [c for c in ["all_linear_off", "all_attention_off", "all_ssm_off"]
                                if c in model_df["condition"].unique()]

        row = {"Model": MODEL_SPECS[model_key]["display_name"]}
        for b in BENCHMARKS_MAIN:
            bname = BENCHMARK_CONFIGS[b]["display_name"]
            row[f"{bname}"] = f"{baseline.get(b, 0):.3f}"
        table_rows.append(row)

        for gc in group_conds_present:
            gc_scores = model_df[model_df["condition"] == gc].set_index("benchmark")["score"]
            row = {"Model": f"  → {gc.replace('all_', '').replace('_off', '').upper()} off"}
            for b in BENCHMARKS_MAIN:
                bname = BENCHMARK_CONFIGS[b]["display_name"]
                base_val = baseline.get(b, 0)
                ablated_val = gc_scores.get(b, 0)
                delta = ablated_val - base_val
                row[f"{bname}"] = f"{ablated_val:.3f} ({delta:+.3f})"
            table_rows.append(row)

    table_df = pd.DataFrame(table_rows)
    save_dataframe(table_df, "paper_table1_group_ablation_delta", index=False)
    display(table_df)
    print("✅ Paper Table 1 (delta from baseline) saved")
except Exception as e:
    print(f"Table 1 skipped: {e}")

print("\n✅ All additional publication figures generated.")


In [ ]:
# --- Cell 3G-pre: Perplexity helper functions ---
# What this cell does:
# 1. Defines compute_next_token_loss for WikiText-2 perplexity evaluation
# 2. Defines evaluate_perplexity_under_ablation for group ablation perplexity
#
# These functions are needed by Cell 3G and Cell 7A.


def build_diagnostic_text():
    """Build diagnostic text from WikiText-2 (defined early for perplexity evaluation)."""
    dataset = get_dataset_cached("wikitext", "wikitext-2-raw-v1", "validation")
    pieces = []
    for ex in dataset:
        txt = ex["text"].strip()
        if len(txt) > 0:
            pieces.append(txt)
        if len(" ".join(pieces)) > 50000:
            break
    return "\n".join(pieces)

def compute_next_token_loss(model, input_ids, attention_mask):
    with torch.inference_mode():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)
        logits = outputs.logits[:, :-1, :]
        labels = input_ids[:, 1:]
        loss = torch.nn.functional.cross_entropy(
            logits.reshape(-1, logits.shape[-1]),
            labels.reshape(-1),
            reduction="mean",
        )
    return float(loss.item())


def evaluate_perplexity_under_ablation(model_key: str):
    exp_name = f"perplexity_ablation__{model_key}"
    cached = ckpt.load(exp_name)
    if cached is not None and cached.get("status") == "complete":
        ckpt.log(f"✅ {exp_name} already completed. Loading from checkpoint.")
        return cached

    arch = get_architecture_result(model_key)["summary"]
    model, tokenizer, meta = load_model_and_tokenizer(model_key)
    try:
        text = build_diagnostic_text()
        max_tokens = 2048 if QUICK_EVAL else 8192
        max_tokens = min(max_tokens, get_max_inference_seq_length(model_key))
        tokenized = tokenizer(text, return_tensors="pt", add_special_tokens=False)
        input_ids = tokenized["input_ids"][:, :max_tokens].to(DEVICE)
        attention_mask = torch.ones_like(input_ids, device=DEVICE)

        rows = []

        # Baseline perplexity
        loss = compute_next_token_loss(model, input_ids, attention_mask)
        clamped_loss = min(loss, 20.0)
        rows.append({"condition": "baseline", "loss": loss, "perplexity": math.exp(clamped_loss)})

        arch_family = MODEL_SPECS[model_key]["arch_family"]

        if arch_family == "qwen_sequential_hybrid":
            with AblationManager(model, model_key, arch) as ablator:
                ablator.skip_all_layers_of_type("linear")
                loss = compute_next_token_loss(model, input_ids, attention_mask)
                clamped_loss = min(loss, 20.0)
                rows.append({"condition": "all_linear_off", "loss": loss, "perplexity": math.exp(clamped_loss)})

            with AblationManager(model, model_key, arch) as ablator:
                ablator.skip_all_layers_of_type("attention")
                loss = compute_next_token_loss(model, input_ids, attention_mask)
                clamped_loss = min(loss, 20.0)
                rows.append({"condition": "all_attention_off", "loss": loss, "perplexity": math.exp(clamped_loss)})
        else:
            with AblationManager(model, model_key, arch) as ablator:
                ablator.zero_component_output("ssm")
                loss = compute_next_token_loss(model, input_ids, attention_mask)
                clamped_loss = min(loss, 20.0)
                rows.append({"condition": "all_ssm_off", "loss": loss, "perplexity": math.exp(clamped_loss)})

            with AblationManager(model, model_key, arch) as ablator:
                ablator.zero_component_output("attention")
                loss = compute_next_token_loss(model, input_ids, attention_mask)
                clamped_loss = min(loss, 20.0)
                rows.append({"condition": "all_attention_off", "loss": loss, "perplexity": math.exp(clamped_loss)})

        payload = {"status": "complete", "model_key": model_key, "rows": rows}
        ckpt.save(exp_name, payload)
        return payload
    finally:
        cleanup_model(model, tokenizer)

print("✅ Perplexity helper functions defined.")


In [ ]:
# --- Cell 3G-extra: Random Control Perplexity + Transformer Baseline ---
# What this cell does:
# 1. Measures perplexity under random layer removal for Qwen3.5-0.8B (5 trials x 2 conditions)
#    to rule out layer-count as a confound for the backbone finding
# 2. Repeats the experiment on a pure Transformer (Qwen2.5-0.5B) as baseline
#    to determine whether redundancy patterns are hybrid-specific
# Requires: compute_perplexity() from Cell 3G-pre, BASE_DIR from Cell 0B
# Time: ~10 minutes on L4/A100
# ==========================================================================

import torch, random, csv, os
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

NUM_TRIALS = 5

def compute_perplexity(model, tokenizer, max_length=512):
    """Compute WikiText-2 perplexity."""
    dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    text = "\n\n".join([t for t in dataset["text"] if t.strip()])
    encodings = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
    input_ids = encodings.input_ids.to(model.device)
    with torch.no_grad():
        outputs = model(input_ids, labels=input_ids)
        loss = outputs.loss.item()
    return loss, torch.exp(torch.tensor(loss)).item()

QWEN_LAYER_TYPES = [
    "linear", "linear", "linear", "attention",   # 0-3
    "linear", "linear", "linear", "attention",   # 4-7
    "linear", "linear", "linear", "attention",   # 8-11
    "linear", "linear", "linear", "attention",   # 12-15
    "linear", "linear", "linear", "attention",   # 16-19
    "linear", "linear", "linear", "attention",   # 20-23
]

# ── Part A: Hybrid random controls (Qwen3.5-0.8B) ──────────────────────
results_hybrid = []
print("Loading Qwen3.5-0.8B (hybrid)...")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-0.8B-Base", trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3.5-0.8B-Base", torch_dtype=torch.bfloat16,
    device_map="auto", trust_remote_code=True
)
model.eval()

for n_layers, label in [(18, "random_control_linear"), (6, "random_control_attention")]:
    for trial in range(NUM_TRIALS):
        random.seed(42 + trial)
        chosen = sorted(random.sample(range(24), n_layers))
        hooks = []
        for idx in chosen:
            layer = model.model.layers[idx]
            hook = layer.register_forward_hook(
                lambda mod, inp, out: (inp[0],) + out[1:] if isinstance(out, tuple) else inp[0]
            )
            hooks.append(hook)
        loss, ppl = compute_perplexity(model, tokenizer)
        for h in hooks:
            h.remove()
        print(f"  {label} trial{trial}: layers={chosen[:5]}... ppl={ppl:.1f}")
        results_hybrid.append({"model": "qwen3.5-0.8b", "condition": f"{label}_trial{trial}",
                               "loss": loss, "perplexity": ppl, "layers_removed": n_layers})

del model, tokenizer; torch.cuda.empty_cache()

# ── Part B: Transformer baseline (Qwen2.5-0.5B) ───────────────────────
results_transformer = []
print("\nLoading Qwen2.5-0.5B (pure Transformer)...")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B", trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B", torch_dtype=torch.bfloat16,
    device_map="auto", trust_remote_code=True
)
model.eval()

loss_base, ppl_base = compute_perplexity(model, tokenizer)
print(f"Baseline: ppl={ppl_base:.1f}")
results_transformer.append({"condition": "baseline", "loss": loss_base,
                             "perplexity": ppl_base, "layers_removed": 0})

for n_layers, label in [(18, "random_18_layers"), (6, "random_6_layers")]:
    for trial in range(NUM_TRIALS):
        random.seed(42 + trial)
        chosen = sorted(random.sample(range(24), n_layers))
        hooks = []
        for idx in chosen:
            layer = model.model.layers[idx]
            hook = layer.register_forward_hook(
                lambda mod, inp, out: (inp[0],) + out[1:] if isinstance(out, tuple) else inp[0]
            )
            hooks.append(hook)
        loss, ppl = compute_perplexity(model, tokenizer)
        for h in hooks:
            h.remove()
        print(f"  {label} trial{trial}: layers={chosen[:5]}... ppl={ppl:.1f}")
        results_transformer.append({"condition": f"{label}_trial{trial}",
                                    "loss": loss, "perplexity": ppl, "layers_removed": n_layers})

del model, tokenizer; torch.cuda.empty_cache()

# ── Save results ───────────────────────────────────────────────────────
for name, data in [("random_control_perplexity.csv", results_hybrid),
                    ("transformer_baseline_perplexity.csv", results_transformer)]:
    path = os.path.join(RESULTS_DIR, name)
    keys = list(data[0].keys())
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys); w.writeheader(); w.writerows(data)
    print(f"Saved {path}")

print("\n✅ Random control perplexity + Transformer baseline complete.")



In [ ]:
# --- Cell 3G: WikiText-2 perplexity under group ablation ---
# What this cell does:
# 1. Evaluates WikiText-2 perplexity under baseline and group ablation conditions
# 2. Produces a bar chart comparing perplexity across conditions
#
# Expected output:
# - perplexity DataFrame
# - bar chart saved to Drive
import gc
perplexity_results = {}
for model_key in MODELS_TO_RUN:
    try:
        perplexity_results[model_key] = evaluate_perplexity_under_ablation(model_key)
    except Exception as exc:
        ckpt.log(f"[ERROR] Perplexity evaluation failed for {model_key}: {exc}")
        traceback.print_exc()
        perplexity_results[model_key] = {"status": "failed", "error": str(exc)}

ppl_rows = []
for model_key, result in perplexity_results.items():
    if isinstance(result, dict) and result.get("status") == "failed":
        continue
    for row in result.get("rows", []):
        ppl_rows.append({"model_key": model_key, **row})

if ppl_rows:
    ppl_df = pd.DataFrame(ppl_rows)
    save_dataframe(ppl_df, "perplexity_under_ablation", index=False)
    display(ppl_df)

    fig, ax = plt.subplots(figsize=(10, 5))
    models_in_ppl = sorted(ppl_df["model_key"].unique())
    conditions_in_ppl = sorted(ppl_df["condition"].unique())
    n_conds = len(conditions_in_ppl)
    bar_width = 0.8 / max(n_conds, 1)
    for i, condition in enumerate(conditions_in_ppl):
        sub = ppl_df[ppl_df["condition"] == condition]
        sub = sub.set_index("model_key").reindex(models_in_ppl)
        ax.bar(np.arange(len(models_in_ppl)) + i * bar_width, sub["perplexity"], width=bar_width, label=condition)
    ax.set_xticks(np.arange(len(models_in_ppl)) + bar_width * (n_conds - 1) / 2)
    ax.set_xticklabels(models_in_ppl)
    ax.set_ylabel("Perplexity (WikiText-2)")
    ax.set_title("WikiText-2 perplexity under group ablation")
    ax.legend()
    ax.set_yscale("log")
    save_figure(fig, "perplexity_under_ablation_bar")
    print("Perplexity evaluation complete.")
else:
    print("No perplexity results available.")

# Section 4 — Experiment 2: layer-wise contribution metrics

This section computes:

1. **hidden-state norm change**
2. **cosine similarity between pre- and post-layer states**
3. **component-output norm ratios**
4. **logit-lens KL divergence** under single-layer or single-component ablation


In [ ]:
# --- Cell 4A: Diagnostic corpus and hidden-state capture helpers ---
# What this cell does:
# 1. Builds a reusable diagnostic token batch from WikiText-2
# 2. Captures layer-level hidden states and component outputs with hooks
# 3. Adds logit-lens projection utilities
#
# Expected output:
# - helper functions defined
# - no model execution yet

def build_diagnostic_text():
    dataset = get_dataset_cached("wikitext", "wikitext-2-raw-v1", "validation")
    pieces = []
    for ex in dataset:
        txt = ex["text"].strip()
        if len(txt) > 0:
            pieces.append(txt)
        if len(" ".join(pieces)) > 50000:
            break
    return "\n".join(pieces)

def build_diagnostic_batch(tokenizer, num_tokens: int = DIAGNOSTIC_NUM_TOKENS):
    text = build_diagnostic_text()
    tokenized = tokenizer(text, return_tensors="pt", add_special_tokens=False)
    input_ids = tokenized["input_ids"][:, :num_tokens].to(DEVICE)
    attention_mask = torch.ones_like(input_ids, device=DEVICE)
    return input_ids, attention_mask

def capture_layer_io(model, input_ids, attention_mask=None):
    layers, _ = get_decoder_layers(model)
    cache = {}
    handles = []

    def make_hook(layer_idx):
        def hook(module, inputs, output):
            pre = inputs[0].detach().float().cpu()
            post = output[0].detach().float().cpu() if isinstance(output, (tuple, list)) else output.detach().float().cpu()
            cache[layer_idx] = {"pre": pre, "post": post}
        return hook

    for i, layer in enumerate(layers):
        handles.append(layer.register_forward_hook(make_hook(i)))

    try:
        with torch.inference_mode():
            _ = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)
    finally:
        for h in handles:
            h.remove()

    return cache

def capture_selected_layer_post_state(model, input_ids, attention_mask, target_layer_idx: int):
    layers, _ = get_decoder_layers(model)
    captured = {}
    handle = None

    def hook(module, inputs, output):
        post = output[0].detach().float().cpu() if isinstance(output, (tuple, list)) else output.detach().float().cpu()
        captured["post"] = post

    handle = layers[target_layer_idx].register_forward_hook(hook)
    try:
        with torch.inference_mode():
            _ = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)
    finally:
        handle.remove()

    return captured["post"]

def compute_norm_change(pre: torch.Tensor, post: torch.Tensor) -> float:
    delta = post - pre
    numerator = torch.norm(delta, dim=-1).mean()
    denominator = torch.norm(pre, dim=-1).mean().clamp_min(1e-8)
    return float((numerator / denominator).item())

def compute_cosine_similarity(pre: torch.Tensor, post: torch.Tensor) -> float:
    pre_f = pre.reshape(-1, pre.shape[-1])
    post_f = post.reshape(-1, post.shape[-1])
    cos = torch.nn.functional.cosine_similarity(pre_f, post_f, dim=-1)
    return float(cos.mean().item())

def project_hidden_to_logits(model, hidden_state: torch.Tensor):
    final_norm = get_final_norm_module(model)
    lm_head = get_lm_head(model)

    hidden_state = hidden_state.to(DEVICE)
    if final_norm is not None:
        hidden_state = final_norm(hidden_state)
    logits = lm_head(hidden_state)
    return logits.float().cpu()

def kl_divergence_from_logits(p_logits: torch.Tensor, q_logits: torch.Tensor) -> float:
    p = torch.softmax(p_logits, dim=-1)
    log_p = torch.log_softmax(p_logits, dim=-1)
    log_q = torch.log_softmax(q_logits, dim=-1)
    kl = torch.sum(p * (log_p - log_q), dim=-1).mean()
    return float(kl.item())

def capture_component_output_norm_ratios(model, model_key: str, architecture_summary: dict, input_ids, attention_mask):
    layers, _ = get_decoder_layers(model)
    layer_input_norms = {}
    rows = []
    handles = []

    def make_pre_hook(layer_idx):
        def pre_hook(module, inputs):
            x = inputs[0].detach().float().cpu()
            layer_input_norms[layer_idx] = float(torch.norm(x, dim=-1).mean().item())
        return pre_hook

    for i, layer in enumerate(layers):
        handles.append(layer.register_forward_pre_hook(make_pre_hook(i)))

    arch_family = MODEL_SPECS[model_key]["arch_family"]

    if arch_family == "qwen_sequential_hybrid":
        for i, layer in enumerate(layers):
            layer_type = architecture_summary["layer_types_norm"][i]
            module = getattr(layer, "linear_attn", None) if layer_type == "linear" else getattr(layer, "self_attn", None)
            if module is None:
                continue

            def make_comp_hook(layer_idx=i, component_type=layer_type):
                def hook(module, inputs, output):
                    tensor = extract_primary_tensor(output)
                    if tensor is None:
                        return
                    out_norm = float(torch.norm(tensor.detach().float().cpu(), dim=-1).mean().item())
                    in_norm = layer_input_norms.get(layer_idx, np.nan)
                    rows.append({
                        "layer_idx": layer_idx,
                        "component_type": component_type,
                        "component_output_norm": out_norm,
                        "layer_input_norm": in_norm,
                        "component_output_norm_ratio": out_norm / max(in_norm, 1e-8),
                    })
                return hook

            handles.append(module.register_forward_hook(make_comp_hook()))
    else:
        for layer_idx, path in architecture_summary["attention_paths"].items():
            module = resolve_module_by_path(model, path)

            def make_comp_hook(layer_idx=layer_idx, component_type="attention"):
                def hook(module, inputs, output):
                    tensor = extract_primary_tensor(output)
                    if tensor is None:
                        return
                    out_norm = float(torch.norm(tensor.detach().float().cpu(), dim=-1).mean().item())
                    in_norm = layer_input_norms.get(layer_idx, np.nan)
                    rows.append({
                        "layer_idx": layer_idx,
                        "component_type": component_type,
                        "component_output_norm": out_norm,
                        "layer_input_norm": in_norm,
                        "component_output_norm_ratio": out_norm / max(in_norm, 1e-8),
                    })
                return hook

            handles.append(module.register_forward_hook(make_comp_hook()))

        for layer_idx, path in architecture_summary["ssm_paths"].items():
            module = resolve_module_by_path(model, path)

            def make_comp_hook(layer_idx=layer_idx, component_type="ssm"):
                def hook(module, inputs, output):
                    tensor = extract_primary_tensor(output)
                    if tensor is None:
                        return
                    out_norm = float(torch.norm(tensor.detach().float().cpu(), dim=-1).mean().item())
                    in_norm = layer_input_norms.get(layer_idx, np.nan)
                    rows.append({
                        "layer_idx": layer_idx,
                        "component_type": component_type,
                        "component_output_norm": out_norm,
                        "layer_input_norm": in_norm,
                        "component_output_norm_ratio": out_norm / max(in_norm, 1e-8),
                    })
                return hook

            handles.append(module.register_forward_hook(make_comp_hook()))

    try:
        with torch.inference_mode():
            _ = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)
    finally:
        for h in handles:
            h.remove()

    return pd.DataFrame(rows)

In [ ]:
# --- Cell 4B: Experiment 2 runner ---
# What this cell does:
# 1. Computes baseline layer metrics from hidden-state captures
# 2. Computes per-layer logit-lens KL divergence under targeted ablations
# 3. Saves metrics tables and checkpoints for each model
#
# Expected output:
# - per-model experiment 2 checkpoints
# - metrics DataFrames written to Drive

def run_experiment2_for_model(model_key: str):
    experiment_name = f"exp2_master__{model_key}"
    cached = ckpt.load(experiment_name)
    if cached is not None and cached.get("status") == "complete":
        ckpt.log(f"✅ {experiment_name} already completed. Loading from checkpoint.")
        return cached

    arch = get_architecture_result(model_key)["summary"]
    model, tokenizer, meta = load_model_and_tokenizer(model_key)

    try:
        input_ids, attention_mask = build_diagnostic_batch(tokenizer, num_tokens=DIAGNOSTIC_NUM_TOKENS)

        # Baseline layer IO
        layer_io = capture_layer_io(model, input_ids, attention_mask)
        base_rows = []
        for layer_idx, tensors in layer_io.items():
            pre = tensors["pre"]
            post = tensors["post"]
            if MODEL_SPECS[model_key]["arch_family"] == "qwen_sequential_hybrid":
                component_type = arch["layer_types_norm"][layer_idx]
            else:
                component_type = "hybrid_block"
            base_rows.append({
                "layer_idx": layer_idx,
                "component_type": component_type,
                "norm_change": compute_norm_change(pre, post),
                "cosine_similarity": compute_cosine_similarity(pre, post),
            })
        base_df = pd.DataFrame(base_rows)

        # Component output norm ratios
        comp_norm_df = capture_component_output_norm_ratios(model, model_key, arch, input_ids, attention_mask)

        # Baseline post-layer hidden states for logit lens
        baseline_post_states = {
            idx: data["post"][:, -1:, :] for idx, data in layer_io.items()
        }
        baseline_logit_lens = {
            idx: project_hidden_to_logits(model, hidden)
            for idx, hidden in baseline_post_states.items()
        }

        kl_rows = []
        num_layers = arch["num_layers"]

        if MODEL_SPECS[model_key]["arch_family"] == "qwen_sequential_hybrid":
            for layer_idx in tqdm(range(num_layers), desc=f"Exp2 logit-lens KL ({model_key})"):
                component_type = arch["layer_types_norm"][layer_idx]
                with AblationManager(model, model_key, arch) as ablator:
                    ablator.skip_layer(layer_idx)
                    ablated_post = capture_selected_layer_post_state(model, input_ids, attention_mask, layer_idx)[:, -1:, :]
                    ablated_logits = project_hidden_to_logits(model, ablated_post)
                baseline_logits = baseline_logit_lens[layer_idx]
                kl_rows.append({
                    "layer_idx": layer_idx,
                    "component_type": component_type,
                    "logit_lens_kl": kl_divergence_from_logits(baseline_logits, ablated_logits),
                })
        else:
            for component_type in ["attention", "ssm"]:
                for layer_idx in tqdm(range(num_layers), desc=f"Exp2 logit-lens KL ({model_key}, {component_type})"):
                    with AblationManager(model, model_key, arch) as ablator:
                        ablator.zero_component_output(component_type, layer_indices=[layer_idx])
                        ablated_post = capture_selected_layer_post_state(model, input_ids, attention_mask, layer_idx)[:, -1:, :]
                        ablated_logits = project_hidden_to_logits(model, ablated_post)
                    baseline_logits = baseline_logit_lens[layer_idx]
                    kl_rows.append({
                        "layer_idx": layer_idx,
                        "component_type": component_type,
                        "logit_lens_kl": kl_divergence_from_logits(baseline_logits, ablated_logits),
                    })

        kl_df = pd.DataFrame(kl_rows)

        metrics_df = base_df.merge(
            comp_norm_df[["layer_idx", "component_type", "component_output_norm_ratio"]],
            on=["layer_idx", "component_type"],
            how="outer",
        ).merge(
            kl_df,
            on=["layer_idx", "component_type"],
            how="outer",
        )

        metrics_csv = os.path.join(RESULTS_DIR, f"{model_key}_experiment2_metrics.csv")
        metrics_df.to_csv(metrics_csv, index=False)

        payload = {
            "status": "complete",
            "model_key": model_key,
            "metrics_df": metrics_df,
            "metrics_csv": metrics_csv,
        }
        ckpt.save(experiment_name, payload)
        return payload
    finally:
        cleanup_model(model, tokenizer)

if "exp2_results" not in globals() or not isinstance(exp2_results, dict) or len(exp2_results) == 0:
    exp2_results = {model_key: ckpt.load(f"exp2_master__{model_key}") for model_key in MODELS_TO_RUN}

In [ ]:
# Fix: patch project_hidden_to_logits to handle dtype mismatch
_original_project = project_hidden_to_logits

def project_hidden_to_logits(model, hidden_state):
    lm_head = get_lm_head(model)
    hidden_state = hidden_state.to(device=next(lm_head.parameters()).device,
                                    dtype=next(lm_head.parameters()).dtype)
    with torch.inference_mode():
        return lm_head(hidden_state)

print("✅ project_hidden_to_logits patched for dtype compatibility")

In [ ]:
# --- Cell 4C: Execute Experiment 2 and generate figures ---
# What this cell does:
# 1. Runs the layer-wise metrics analysis for each model
# 2. Generates publication-style line plots and heatmaps
#
# Expected output:
# - experiment 2 result tables
# - figures saved to Drive

exp2_results = {}

for model_key in MODELS_TO_RUN:
    try:
        exp2_results[model_key] = run_experiment2_for_model(model_key)
    except Exception as exc:
        ckpt.log(f"[ERROR] Experiment 2 failed for {model_key}: {exc}")
        traceback.print_exc()
        exp2_results[model_key] = {"status": "failed", "error": str(exc)}

for model_key, result in exp2_results.items():
    if result.get("status") == "failed":
        continue

    metrics_df = result["metrics_df"].sort_values(["component_type", "layer_idx"])
    save_dataframe(metrics_df, f"{model_key}_experiment2_metrics", index=False)

    # Norm change
    fig, ax = plt.subplots(figsize=(10, 4))
    for component_type, sub in metrics_df.groupby("component_type"):
        if "norm_change" in sub.columns and sub["norm_change"].notna().any():
            ax.plot(sub["layer_idx"], sub["norm_change"], marker="o", label=component_type)
    ax.set_xlabel("Layer index")
    ax.set_ylabel("||h_post - h_pre|| / ||h_pre||")
    ax.set_title(f"{model_key}: hidden-state norm change by layer")
    ax.legend()
    save_figure(fig, f"{model_key}_exp2_norm_change")

    # Cosine similarity
    fig, ax = plt.subplots(figsize=(10, 4))
    for component_type, sub in metrics_df.groupby("component_type"):
        if "cosine_similarity" in sub.columns and sub["cosine_similarity"].notna().any():
            ax.plot(sub["layer_idx"], sub["cosine_similarity"], marker="o", label=component_type)
    ax.set_xlabel("Layer index")
    ax.set_ylabel("cosine(h_pre, h_post)")
    ax.set_title(f"{model_key}: cosine similarity by layer")
    ax.legend()
    save_figure(fig, f"{model_key}_exp2_cosine_similarity")

    # Component output norm ratio
    if metrics_df["component_output_norm_ratio"].notna().any():
        fig, ax = plt.subplots(figsize=(10, 4))
        for component_type, sub in metrics_df.groupby("component_type"):
            sub = sub[sub["component_output_norm_ratio"].notna()]
            if len(sub) == 0:
                continue
            ax.plot(sub["layer_idx"], sub["component_output_norm_ratio"], marker="o", label=component_type)
        ax.set_xlabel("Layer index")
        ax.set_ylabel("component output norm / layer input norm")
        ax.set_title(f"{model_key}: component output norm ratio")
        ax.legend()
        save_figure(fig, f"{model_key}_exp2_component_output_norm_ratio")

    # Logit-lens KL
    kl_df = metrics_df[metrics_df["logit_lens_kl"].notna()].copy()
    if len(kl_df) > 0:
        fig, ax = plt.subplots(figsize=(10, 4))
        for component_type, sub in kl_df.groupby("component_type"):
            ax.plot(sub["layer_idx"], sub["logit_lens_kl"], marker="o", label=component_type)
        ax.set_xlabel("Layer index")
        ax.set_ylabel("KL(baseline || ablated)")
        ax.set_title(f"{model_key}: logit-lens divergence by layer")
        ax.legend()
        save_figure(fig, f"{model_key}_exp2_logit_lens_kl")

print("Experiment 2 complete.")

In [ ]:
# --- Cell 4C-extra: Correlation between Exp 2 metrics and Exp 1 accuracy drop ---
# What this cell does:
# 1. Correlates layer-wise metrics (norm change, logit-lens KL) with ablation accuracy drop
# 2. Generates scatter plots with regression lines
# 3. This figure CONNECTS Exp 1 and Exp 2 — key for the paper narrative
#
# Expected output:
# - Scatter plots showing whether layers that contribute more (Exp 2)
#   also cause more degradation when ablated (Exp 1)

import seaborn as sns
from scipy import stats

try:
    fig, axes = plt.subplots(len(MODELS_TO_RUN), 2, figsize=(12, 5 * len(MODELS_TO_RUN)), squeeze=False)

    for row_idx, model_key in enumerate(sorted(MODELS_TO_RUN)):
        exp2_result = exp2_results.get(model_key)
        if exp2_result is None or exp2_result.get("status") == "failed":
            continue

        metrics_df = exp2_result["metrics_df"] if isinstance(exp2_result.get("metrics_df"), pd.DataFrame) else pd.DataFrame()
        if len(metrics_df) == 0 and "payload" in exp2_result:
            metrics_df = exp2_result["payload"].get("metrics_df", pd.DataFrame())
        if len(metrics_df) == 0:
            continue

        model_exp1 = exp1_df[exp1_df["model_key"] == model_key]
        baseline_scores = model_exp1[model_exp1["condition"] == "baseline"].set_index("benchmark")["score"]

        # Compute avg accuracy drop per layer from Exp 1 layer sweep
        arch_family = MODEL_SPECS[model_key]["arch_family"]
        component_types = ["ssm", "attention"] if "falcon" in arch_family else ["linear", "attention"]

        merged_rows = []
        for _, mrow in metrics_df.iterrows():
            layer_idx = mrow["layer_idx"]
            comp = mrow["component_type"]

            # Find matching Exp 1 condition
            if comp == "hybrid_block":
                # Falcon: try both ssm and attention layer conditions
                for ct in ["ssm", "attention"]:
                    cond_name = f"{ct}_layer_{layer_idx:02d}_off"
                    cond_df = model_exp1[model_exp1["condition"] == cond_name]
                    if len(cond_df) > 0:
                        cond_scores = cond_df.set_index("benchmark")["score"]
                        common = set(baseline_scores.index) & set(cond_scores.index)
                        if common:
                            avg_drop = np.mean([baseline_scores[b] - cond_scores[b] for b in common])
                            merged_rows.append({
                                "layer_idx": layer_idx,
                                "component": ct,
                                "norm_change": mrow.get("norm_change", np.nan),
                                "logit_lens_kl": mrow.get("logit_lens_kl", np.nan),
                                "accuracy_drop": avg_drop,
                            })
            else:
                cond_name = f"{comp}_layer_{layer_idx:02d}_off" if "falcon" in arch_family else f"layer_{layer_idx:02d}_{comp}_off"
                # Try alternate naming
                for candidate in [cond_name, f"layer_{layer_idx:02d}_{comp}_off",
                                  f"{comp}_layer_{layer_idx:02d}_off"]:
                    cond_df = model_exp1[model_exp1["condition"] == candidate]
                    if len(cond_df) > 0:
                        cond_scores = cond_df.set_index("benchmark")["score"]
                        common = set(baseline_scores.index) & set(cond_scores.index)
                        if common:
                            avg_drop = np.mean([baseline_scores[b] - cond_scores[b] for b in common])
                            merged_rows.append({
                                "layer_idx": layer_idx,
                                "component": comp,
                                "norm_change": mrow.get("norm_change", np.nan),
                                "logit_lens_kl": mrow.get("logit_lens_kl", np.nan),
                                "accuracy_drop": avg_drop,
                            })
                        break

        if not merged_rows:
            continue

        merged_df = pd.DataFrame(merged_rows)

        # Plot 1: norm_change vs accuracy_drop
        ax1 = axes[row_idx, 0]
        for comp, color in [("ssm", "#FF9800"), ("attention", "#2196F3"),
                            ("linear", "#FF9800"), ("hybrid_block", "#9E9E9E")]:
            sub = merged_df[merged_df["component"] == comp]
            if len(sub) > 0 and sub["norm_change"].notna().any():
                ax1.scatter(sub["norm_change"], sub["accuracy_drop"], c=color,
                           label=comp, alpha=0.7, edgecolors="black", linewidth=0.3, s=30)
        # Regression line (all points)
        valid = merged_df.dropna(subset=["norm_change", "accuracy_drop"])
        if len(valid) > 2:
            r, p = stats.pearsonr(valid["norm_change"], valid["accuracy_drop"])
            z = np.polyfit(valid["norm_change"], valid["accuracy_drop"], 1)
            xline = np.linspace(valid["norm_change"].min(), valid["norm_change"].max(), 50)
            ax1.plot(xline, np.polyval(z, xline), "k--", linewidth=1, alpha=0.5)
            ax1.set_xlabel("Norm change (Exp 2)")
            ax1.set_ylabel("Avg accuracy drop (Exp 1)")
            ax1.set_title(f"{MODEL_SPECS[model_key]['display_name']}\nr={r:.3f}, p={p:.3f}")
            ax1.legend(fontsize=8)

        # Plot 2: logit_lens_kl vs accuracy_drop
        ax2 = axes[row_idx, 1]
        for comp, color in [("ssm", "#FF9800"), ("attention", "#2196F3"),
                            ("linear", "#FF9800"), ("hybrid_block", "#9E9E9E")]:
            sub = merged_df[merged_df["component"] == comp]
            if len(sub) > 0 and sub["logit_lens_kl"].notna().any():
                ax2.scatter(sub["logit_lens_kl"], sub["accuracy_drop"], c=color,
                           label=comp, alpha=0.7, edgecolors="black", linewidth=0.3, s=30)
        valid = merged_df.dropna(subset=["logit_lens_kl", "accuracy_drop"])
        if len(valid) > 2:
            r, p = stats.pearsonr(valid["logit_lens_kl"], valid["accuracy_drop"])
            z = np.polyfit(valid["logit_lens_kl"], valid["accuracy_drop"], 1)
            xline = np.linspace(valid["logit_lens_kl"].min(), valid["logit_lens_kl"].max(), 50)
            ax2.plot(xline, np.polyval(z, xline), "k--", linewidth=1, alpha=0.5)
            ax2.set_xlabel("Logit-lens KL divergence (Exp 2)")
            ax2.set_ylabel("Avg accuracy drop (Exp 1)")
            ax2.set_title(f"{MODEL_SPECS[model_key]['display_name']}\nr={r:.3f}, p={p:.3f}")
            ax2.legend(fontsize=8)

    fig.suptitle("Do important layers (Exp 2) cause more damage when ablated (Exp 1)?", fontsize=13, y=1.02)
    plt.tight_layout()
    save_figure(fig, "paper_exp1_exp2_correlation")
    print("✅ Exp 1 ↔ Exp 2 correlation plot saved")
except Exception as e:
    print(f"Correlation plot skipped: {e}")
    import traceback; traceback.print_exc()


# Section 5 — Experiment 3: task-dependent analysis

This section turns Experiment 1 into a benchmark × ablated-component × model analysis table and adds simple paired bootstrap intervals for the score drop.


In [ ]:
# --- Cell 5A: Statistical helpers and Experiment 3 aggregation ---
# What this cell does:
# 1. Aligns baseline and ablated predictions by example_id
# 2. Computes paired bootstrap confidence intervals for accuracy drop
# 3. Builds the final task-dependent analysis table
#
# Expected output:
# - experiment 3 summary table
# - LaTeX / CSV exports

def align_predictions_by_id(preds_a: list[dict], preds_b: list[dict]):
    a_map = {x["example_id"]: x for x in preds_a}
    b_map = {x["example_id"]: x for x in preds_b}
    shared = sorted(set(a_map) & set(b_map))
    a = np.array([a_map[k]["correct"] for k in shared], dtype=np.float32)
    b = np.array([b_map[k]["correct"] for k in shared], dtype=np.float32)
    return shared, a, b

def paired_bootstrap_accuracy_drop(baseline_correct, ablated_correct, n_boot: int = 1000, seed: int = SEED):
    rng = np.random.default_rng(seed)
    n = len(baseline_correct)
    if n == 0:
        return {"mean_drop": None, "ci_low": None, "ci_high": None}
    diffs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        diffs.append(float(np.mean(baseline_correct[idx] - ablated_correct[idx])))
    diffs = np.array(diffs)
    return {
        "mean_drop": float(np.mean(baseline_correct - ablated_correct)),
        "ci_low": float(np.quantile(diffs, 0.025)),
        "ci_high": float(np.quantile(diffs, 0.975)),
    }

def collect_eval_checkpoint(model_key: str, condition: str, benchmark: str):
    ckpt_name = f"eval__{model_key}__{condition}__{benchmark}"
    return ckpt.load(ckpt_name)

exp3_rows = []

for model_key in MODELS_TO_RUN:
    baseline_conditions = collect_eval_checkpoint(model_key, "baseline", BENCHMARKS_MAIN[0])
    if baseline_conditions is None:
        continue

    if MODEL_SPECS[model_key]["arch_family"] == "qwen_sequential_hybrid":
        comparison_conditions = ["all_linear_off", "all_attention_off"]
    else:
        comparison_conditions = ["all_ssm_off", "all_attention_off"]

    for benchmark in BENCHMARKS_MAIN:
        baseline_result = collect_eval_checkpoint(model_key, "baseline", benchmark)
        if baseline_result is None:
            continue

        for condition in comparison_conditions:
            ablated_result = collect_eval_checkpoint(model_key, condition, benchmark)
            if ablated_result is None:
                continue

            metric_name = BENCHMARK_CONFIGS[benchmark]["metric_name"]
            baseline_score = baseline_result["summary"][metric_name]
            ablated_score = ablated_result["summary"][metric_name]

            shared_ids, base_corr, abl_corr = align_predictions_by_id(
                baseline_result["predictions"], ablated_result["predictions"]
            )
            stats = paired_bootstrap_accuracy_drop(base_corr, abl_corr)

            exp3_rows.append({
                "model_key": model_key,
                "benchmark": benchmark,
                "condition": condition,
                "baseline_score": baseline_score,
                "ablated_score": ablated_score,
                "score_drop": baseline_score - ablated_score,
                "paired_n": len(shared_ids),
                "bootstrap_mean_drop": stats["mean_drop"],
                "bootstrap_ci_low": stats["ci_low"],
                "bootstrap_ci_high": stats["ci_high"],
                "ci_excludes_zero": int((stats["ci_low"] is not None) and (stats["ci_low"] > 0)),
            })

exp3_df = pd.DataFrame(exp3_rows)
save_dataframe(exp3_df, "experiment3_task_dependent_analysis", index=False)
display(exp3_df)

In [ ]:
# --- Cell 5B: Experiment 3 cross-table visualization ---
# What this cell does:
# 1. Creates a benchmark × condition table per model
# 2. Saves a compact visualization for the paper draft
#
# Expected output:
# - pivot table printed
# - figure saved to Drive

if len(exp3_df) > 0:
    cross_df = exp3_df.pivot_table(
        index=["model_key", "condition"],
        columns="benchmark",
        values="score_drop",
    )
    save_dataframe(cross_df.reset_index(), "experiment3_score_drop_pivot", index=False)
    display(cross_df)

    fig, ax = plt.subplots(figsize=(11, 4.5))
    heatmap_df = exp3_df.pivot_table(index=["model_key", "condition"], columns="benchmark", values="score_drop")
    im = ax.imshow(heatmap_df.values, aspect="auto")
    ax.set_xticks(np.arange(len(heatmap_df.columns)))
    ax.set_xticklabels(list(heatmap_df.columns), rotation=30, ha="right")
    ax.set_yticks(np.arange(len(heatmap_df.index)))
    ax.set_yticklabels([f"{m} | {c}" for m, c in heatmap_df.index])
    ax.set_title("Experiment 3: score drop by benchmark and ablated component")
    fig.colorbar(im, ax=ax)
    save_figure(fig, "experiment3_score_drop_heatmap")

print("Experiment 3 complete.")

# Section 6 — Publication-quality analysis and LaTeX exports

This section creates paper-ready summary tables and a concise narrative artifact based on the saved results.


In [ ]:
if "exp1_df" not in globals() or exp1_df is None:
    try:
        exp1_df = pd.read_csv(os.path.join(RESULTS_DIR, "experiment1_summary.csv"))
    except Exception:
        exp1_df = pd.DataFrame()

if "exp3_df" not in globals() or exp3_df is None:
    try:
        exp3_df = pd.read_csv(os.path.join(RESULTS_DIR, "experiment3_task_dependent_analysis.csv"))
    except Exception:
        exp3_df = pd.DataFrame()

if "exp2_results" not in globals() or not isinstance(exp2_results, dict) or len(exp2_results) == 0:
    exp2_results = {model_key: ckpt.load(f"exp2_master__{model_key}") for model_key in MODELS_TO_RUN}

if "exp2_results" not in globals() or not isinstance(exp2_results, dict) or len(exp2_results) == 0:
    exp2_results = {model_key: ckpt.load(f"exp2_master__{model_key}") for model_key in MODELS_TO_RUN}

In [ ]:
# --- Cell 6A: Paper-ready summary tables and concise findings export ---
# What this cell does:
# 1. Creates paper-oriented tables from Experiments 1–3
# 2. Writes a concise JSON / text narrative to Drive
#
# Expected output:
# - summary tables saved
# - final findings text file saved

paper_notes = []

if len(exp1_df) > 0:
    baseline_df = exp1_df[exp1_df["condition"] == "baseline"].copy()
    save_dataframe(baseline_df, "paper_table_baselines", index=False)

if len(exp3_df) > 0:
    for model_key in sorted(exp3_df["model_key"].unique()):
        sub = exp3_df[exp3_df["model_key"] == model_key].copy()
        if len(sub) == 0:
            continue
        strongest = sub.sort_values("score_drop", ascending=False).head(3)
        weakest = sub.sort_values("score_drop", ascending=True).head(3)
        paper_notes.append({
            "model_key": model_key,
            "largest_score_drops": strongest[["benchmark", "condition", "score_drop"]].to_dict(orient="records"),
            "smallest_score_drops": weakest[["benchmark", "condition", "score_drop"]].to_dict(orient="records"),
        })

if len(exp2_results) > 0:
    for model_key, result in exp2_results.items():
        if result.get("status") == "failed":
            continue
        metrics_df = result["metrics_df"]
        if "logit_lens_kl" in metrics_df.columns and metrics_df["logit_lens_kl"].notna().any():
            top_kl = metrics_df.sort_values("logit_lens_kl", ascending=False).head(5)
            paper_notes.append({
                "model_key": model_key,
                "top_logit_lens_kl_layers": top_kl[["layer_idx", "component_type", "logit_lens_kl"]].to_dict(orient="records"),
            })

paper_notes_path = os.path.join(RESULTS_DIR, "paper_ready_findings.json")
save_json_file(paper_notes, paper_notes_path)

summary_txt = os.path.join(RESULTS_DIR, "paper_ready_findings.txt")
lines = [
    "Paper-ready findings summary",
    "=" * 80,
]
for item in paper_notes:
    lines.append(json.dumps(item, indent=2))
save_text("\n\n".join(lines), summary_txt)

print(f"Saved: {paper_notes_path}")
print(f"Saved: {summary_txt}")

# Section 7 — Export results and final summary

This final section creates a compact manifest of all major artifacts written to Google Drive.


In [ ]:
# --- Cell 7A: Optional appendix experiment — context-length stress test ---
# What this cell does:
# 1. Provides an easy complementary experiment for the appendix
# 2. Measures next-token loss at increasing context lengths under group ablation
# 3. Is disabled by default to save compute
#
# Expected output:
# - only runs if RUN_OPTIONAL_CONTEXT_STRESS = True
# Note: compute_next_token_loss and evaluate_perplexity_under_ablation
# are already defined in Cell 3G-pre above.

def run_optional_context_stress_for_model(model_key: str):
    exp_name = f"optional_context_stress__{model_key}"
    cached = ckpt.load(exp_name)
    if cached is not None and cached.get("status") == "complete":
        ckpt.log(f"✅ {exp_name} already completed. Loading from checkpoint.")
        return cached

    arch = get_architecture_result(model_key)["summary"]
    model, tokenizer, meta = load_model_and_tokenizer(model_key)
    try:
        full_ids, full_mask = build_diagnostic_batch(tokenizer, num_tokens=min(max(2048, DIAGNOSTIC_NUM_TOKENS), get_max_inference_seq_length(model_key)))
        lengths = [128, 256, 512, 1024]
        lengths = [x for x in lengths if x <= full_ids.shape[1]]
        rows = []

        # baseline
        for L in lengths:
            ids = full_ids[:, :L]
            mask = full_mask[:, :L]
            rows.append({
                "context_length": L,
                "condition": "baseline",
                "loss": compute_next_token_loss(model, ids, mask),
            })

        if MODEL_SPECS[model_key]["arch_family"] == "qwen_sequential_hybrid":
            with AblationManager(model, model_key, arch) as ablator:
                ablator.skip_all_layers_of_type("linear")
                for L in lengths:
                    ids = full_ids[:, :L]
                    mask = full_mask[:, :L]
                    rows.append({
                        "context_length": L,
                        "condition": "all_linear_off",
                        "loss": compute_next_token_loss(model, ids, mask),
                    })
        else:
            with AblationManager(model, model_key, arch) as ablator:
                ablator.zero_component_output("ssm")
                for L in lengths:
                    ids = full_ids[:, :L]
                    mask = full_mask[:, :L]
                    rows.append({
                        "context_length": L,
                        "condition": "all_ssm_off",
                        "loss": compute_next_token_loss(model, ids, mask),
                    })

        payload = {"status": "complete", "rows": rows}
        ckpt.save(exp_name, payload)
        return payload
    finally:
        cleanup_model(model, tokenizer)

if RUN_OPTIONAL_CONTEXT_STRESS:
    optional_rows = []
    for model_key in MODELS_TO_RUN:
        result = run_optional_context_stress_for_model(model_key)
        for row in result["rows"]:
            row["model_key"] = model_key
            optional_rows.append(row)
    optional_df = pd.DataFrame(optional_rows)
    save_dataframe(optional_df, "optional_context_stress", index=False)
    display(optional_df)
else:
    print("RUN_OPTIONAL_CONTEXT_STRESS = False; skipping optional appendix experiment.")


In [ ]:
# --- Cell 7B: Optional lm-eval cross-check helper ---
# What this cell does:
# 1. Provides a baseline-only harness cross-check
# 2. Does NOT drive the main paper pipeline, because manual evaluation is easier with active hooks
#
# Expected output:
# - only runs if RUN_LM_EVAL_CROSSCHECK = True

def run_lm_eval_baseline_crosscheck(model_key: str, tasks: str = "arc_challenge,hellaswag"):
    if not RUN_LM_EVAL_CROSSCHECK:
        print("RUN_LM_EVAL_CROSSCHECK = False; skipping lm-eval baseline cross-check.")
        return None

    import subprocess
    spec = MODEL_SPECS[model_key]
    model_id = spec["model_id"]
    output_path = os.path.join(RESULTS_DIR, f"lm_eval__{model_key}.json")

    cmd = [
        sys.executable, "-m", "lm_eval",
        "--model", "hf",
        "--model_args", f"pretrained={model_id},dtype={'bfloat16' if DTYPE == torch.bfloat16 else 'float16'}",
        "--tasks", tasks,
        "--device", DEVICE,
        "--batch_size", "1",
        "--output_path", output_path,
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    print(f"Saved lm-eval output to {output_path}")
    return output_path

if RUN_LM_EVAL_CROSSCHECK:
    for model_key in MODELS_TO_RUN:
        try:
            run_lm_eval_baseline_crosscheck(model_key)
        except Exception as exc:
            ckpt.log(f"[WARNING] lm-eval cross-check failed for {model_key}: {exc}")
else:
    print("lm-eval cross-check helper is defined and skipped by default.")

In [ ]:
# --- Cell 7C: Final export manifest and end-to-end runner ---
# What this cell does:
# 1. Creates a compact manifest of generated artifacts
# 2. Optionally runs the whole pipeline from one cell if desired
#
# Expected output:
# - final summary JSON written to Drive
# - a printed manifest table
from glob import glob
RUN_FULL_PIPELINE = False  # set True only when you are ready to launch everything sequentially

def build_artifact_manifest():
    manifest = {
        "timestamp": now_utc(),
        "base_dir": BASE_DIR,
        "results": sorted(glob(os.path.join(RESULTS_DIR, "*"))),
        "figures": sorted(glob(os.path.join(FIGURES_DIR, "*"))),
        "tables": sorted(glob(os.path.join(TABLES_DIR, "*"))),
        "checkpoints": sorted(glob(os.path.join(CHECKPOINTS_DIR, "*.pkl"))),
        "logs": sorted(glob(os.path.join(LOGS_DIR, "*"))),
        "artifacts": sorted(glob(os.path.join(ARTIFACTS_DIR, "*"))),
    }
    manifest["counts"] = {k: len(v) for k, v in manifest.items() if isinstance(v, list)}
    return manifest

if RUN_FULL_PIPELINE:
    architecture_results = {}
    for model_key in MODELS_TO_RUN:
        architecture_results[model_key] = run_architecture_exploration(model_key)

    exp1_results = {}
    for model_key in MODELS_TO_RUN:
        exp1_results[model_key] = run_experiment1_for_model(model_key)

    exp2_results = {}
    for model_key in MODELS_TO_RUN:
        exp2_results[model_key] = run_experiment2_for_model(model_key)

manifest = build_artifact_manifest()
manifest_path = os.path.join(RESULTS_DIR, "final_manifest.json")
save_json_file(manifest, manifest_path)

summary_rows = []
for key, value in manifest["counts"].items():
    summary_rows.append({"artifact_group": key, "count": value})
summary_df = pd.DataFrame(summary_rows)

print(f"Final manifest saved to: {manifest_path}")
display(summary_df)

## End of notebook

### Execution order
1. Run **Section 0** (setup + merge Falcon results)
2. Run **Section 1** (architecture discovery)
3. Run **Section 2** (ablation mechanism)
4. Run **Section 3** (Experiment 1 + perplexity + random controls + Transformer baseline)
5. Run **Section 4** (Experiment 2)
6. Run **Section 5** (Experiment 3)
7. Run **Section 6** (LaTeX tables)
8. Run **Section 7** (manifest + optional appendix)

All outputs are saved to `BASE_DIR/results/`, `BASE_DIR/figures/`, and `BASE_DIR/tables/`.
